# Focussing optics: the focal field of a lens (FocussingOptics)

Plots and animations behind the focussing-optics part of the thesis, made with
`levisim.physics.focussing_optics.FocussingOptics`: the Bessel functions and the integrals
$\mathrm{I}_{00}, \mathrm{I}_{01}, \mathrm{I}_{02}$ of the focal field (in the focal plane and along the
propagation axis), their dependence on focal length, numerical aperture and filling factor, beam
widths, and field maps in the focal region.

**Runtime:** a few minutes with `MAKE_ANIMATIONS = False`, where every animation computes only 4 evenly
spaced frames and shows them overlaid in one static figure. `MAKE_ANIMATIONS = True` computes every
frame and writes the animations to `output/` (much longer).

**Updated Nov 2026:**

- the `colour` package (`pip install colour`, not colour-science) was used for one red→purple gradient;
  it is replaced by `hsl_gradient` below, which interpolates in HSL as `Color.range_to` did;
- animations are no longer embedded as HTML (the stored notebook was 502 MB, almost all of it
  animation frames); they are written to `output/` by `save_animation`;
- an unbalanced brace in the NA axis label (a mathtext parse error in current Matplotlib) was fixed;
- the unit `V` → `Volt` (renamed in `levisim.core.units`);
- the inline xmgrace-style rcParams → `set_plt_rcParams_to_xmgrace_style()`;
- `ContourSet.collections` (removed in Matplotlib 3.10) → the `ContourSet` itself, in the field-map animation;
- log-scale contour maps take their levels from the positive values only (`log_levels`): a
  component that vanishes in a plane by symmetry made the levels NaN, an error in current Matplotlib;
- the random text positions in the last animation are seeded, and its colour `magenta` (no
  longer in `levisim.core.colours`) → `RGB_magenta`;
- a copied Stack Overflow contour-animation example (not levisim code, and broken on current
  Matplotlib) was removed.

In [ ]:
from pathlib import Path

import colorsys
import numpy as np
from matplotlib.animation import writers
from matplotlib.colors import to_hex, to_rgb

# MAKE_ANIMATIONS = True computes every frame and writes the animations to output/ (mp4 with ffmpeg,
# else gif). False (the smoke-test setting) computes 4 evenly spaced frames per animation and shows
# them as one static figure
MAKE_ANIMATIONS = False
OUTPUT_DIR = Path('output')

# the RSVP animation places its text at random
np.random.seed(0)


def preview_frames(values, num=4):
    """All `values` when MAKE_ANIMATIONS, else `num` evenly spaced ones (first and last included)"""
    values = list(values)
    if MAKE_ANIMATIONS or len(values) <= num:
        return values
    indices = np.unique(np.linspace(0, len(values) - 1, num).round().astype(int))
    return [values[ii] for ii in indices]


def save_animation(animation, name):
    """Writes `animation` to output/<name>.mp4 (or .gif without ffmpeg), never into the notebook"""
    OUTPUT_DIR.mkdir(exist_ok=True)
    writer = 'ffmpeg' if writers.is_available('ffmpeg') else 'pillow'
    path = OUTPUT_DIR / f'{name}.{"mp4" if writer == "ffmpeg" else "gif"}'
    animation.save(path, writer=writer)
    print(f'Wrote {path}')
    return path


def log_levels(values, num):
    """
    `num` logarithmically spaced contour levels spanning the positive `values`

    Zeros (a field component that vanishes in this plane by symmetry, e.g. E_y in the xz plane)
    would make log10(min) = -inf and the levels NaN. An all-zero map gets levels just above 0.
    """
    positive = values[values > 0]
    if positive.size == 0:
        return np.logspace(-300, -299, num)
    return np.logspace(np.log10(positive.min()), np.log10(positive.max()), num)


def hsl_gradient(start, end, num):
    """
    `num` hex colours from `start` to `end`, interpolated linearly in hue, saturation and
    lightness - what `colour.Color(start).range_to(colour.Color(end), num)` from the `colour`
    package did, without the dependency
    """
    h0, l0, s0 = colorsys.rgb_to_hls(*to_rgb(start))
    h1, l1, s1 = colorsys.rgb_to_hls(*to_rgb(end))
    return [
        to_hex(colorsys.hls_to_rgb(h0 + t * (h1 - h0), l0 + t * (l1 - l0), s0 + t * (s1 - s0)))
        for t in np.linspace(0, 1, num)
    ]

## Imports and plotting parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from copy import copy
from functools import wraps
from math import factorial, gamma

from scipy.special import jv as Bessel_of_first_kind
from scipy.interpolate import UnivariateSpline

from matplotlib import cm as colourmap
from matplotlib import ticker
from matplotlib.patches import Circle, Ellipse
from matplotlib.colors import LogNorm

from matplotlib.animation import ArtistAnimation

from levisim import *


# match the width (inches) of my thesis document
# commands in latex
# textwidth in cm: \printinunitsof{cm}\prntlen{\textwidth}
# textwidth in in: \printinunitsof{in}\prntlen{\textwidth}
THESIS_DOCUMENT_WIDTH_CM = 13.998  # centimetres
THESIS_DOCUMENT_WIDTH_INCHES = 5.5129  # inches
FS = THESIS_DOCUMENT_WIDTH_INCHES

# levisim's xmgrace-inspired style (the same settings the original defined inline)
set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0

FocussingOptics().info

$$
    J_{n}( \kappa )
    \equiv
    \sum _{j=0}^{\infty }
    \frac{( -1)^{j}}{j!\ \Gamma ( n+1+j)}\left(\frac{\kappa}{2}\right)^{n+2j}
$$


# `FocussingOptics` investigation of beam widths

## Official `scipy` Bessel function vs. version in thesis (up to 100 terms)

In [ ]:
n_max = 5
lim = 10
x = np.linspace(-lim, lim, 500)

Jn = Bessel_of_first_kind
fig, ax = plt.subplots(figsize=(FS, 3), dpi=250)
for i in range(n_max):
    ax.plot(x, Jn(i, x), label=f'$J_{i!r}(\kappa)$', lw=3 - 0.45 * i, alpha=0.7)
    ax.set_xlabel(r'$\kappa$')
ax.hlines(0, -1.1*lim, 1.1*lim, color=grey, ls='--', lw=0.8)
ax.vlines(0, -1.2, 1.2, color=grey, ls='--', lw=0.8)
ax.set_xlim(-lim, lim)
ax.set_ylim(-1.1, 1.1)
ax.legend()
plt.show()

def my_bessel(n, k):
    return np.sum([
        (-1)**jj / (factorial(jj) * gamma(n + 1 + jj)) * (k / 2)**(n + 2*jj)
        for jj in range(100)
    ])

fig, ax = plt.subplots(figsize=(FS, 3), dpi=250)
for i in range(n_max):
    ax.plot(x, [my_bessel(i, xx) for xx in x], label=f'$J_{i!r}(\kappa)$', lw=3 - 0.45 * i, alpha=0.7)
    ax.set_xlabel(r'$\kappa$')
ax.hlines(0, -1.1*lim, 1.1*lim, color=grey, ls='--', lw=0.8)
ax.vlines(0, -1.2, 1.2, color=grey, ls='--', lw=0.8)
ax.set_xlim(-lim, lim)
ax.set_ylim(-1.1, 1.1)
ax.legend()
plt.show()

$$
    \text{NA} \equiv n \sin \theta_{\text{max}}
$$

$$
    \kappa \equiv k \rho \sin \theta
$$

$$
    \mathrm{I}_{00}( \rho ,z)
    \equiv
    \int _{0}^{\theta _{\text{max}}}
    J_{0}( \kappa )
    e^{-f^{2}\sin^{2} \theta /w_{0}^{2}}\sqrt{\cos \theta }
    e^{ikz\cos \theta }
    \sin \theta ( 1+\cos \theta )
    \
    d\theta
$$

$$
    \mathrm{I}_{01}( \rho ,z)
    \equiv
    \int _{0}^{\theta _{\text{max}}}
    J_{1}( \kappa )
    e^{-f^{2}\sin^{2} \theta /w_{0}^{2}}\sqrt{\cos \theta }
    e^{ikz\cos \theta }
    \sin^{2} \theta
    \
    d\theta
$$

$$
    \mathrm{I}_{02}( \rho ,z)
    \equiv
    \int _{0}^{\theta _{\text{max}}}
    J_{2}( \kappa )
    e^{-f^{2}\sin^{2}\theta /w_{0}^{2}}\sqrt{\cos \theta }
    e^{ikz\cos \theta }
    \sin \theta ( 1-\cos \theta )
    \
    d\theta
$$

where there is an implied $\theta \leftarrow \theta/\Theta$ (where $\Theta \equiv 2\pi$ radians $= 360 \degree$ etc.) for the integrals to be dimensionless quantities


$$
    E_0 = \sqrt{\frac{4P}{\pi w_0^2 \varepsilon_0 c}} \ \ \ [\mathrm{V}/\mathrm{m}]
$$

Wave impedance of the medium
$$
Z_{\text{imp.}} \equiv \sqrt{\frac{\mu}{\varepsilon}} = \sqrt{\frac{\mu_0 \mu_r}{\varepsilon_0 \varepsilon_r}} \ \ \ [\mathrm{\Omega}]
$$

(0,0) Gaussian mode
$$
    \mathbf{E}_{\text{focus},00}( \rho ,\varphi ,z)
    =
    -\frac{ikf}{2} E_{0} e^{-ikf}
    \begin{bmatrix}
        \mathrm{I}_{00} +\mathrm{I}_{02}\cos( 2\varphi )\\
        \mathrm{I}_{02}\sin( 2\varphi )\\
        -2i\ \mathrm{I}_{01}\cos (\varphi)
    \end{bmatrix}
    \equiv
    \varSigma \cdot \mathbf{I}^E_{(0, 0)}
$$

$$
    \mathbf{H}_{\text{focus},00}( \rho ,\varphi ,z)
    =
    \frac{1}{Z_{\text{imp.}}}
    \cdot
    -\frac{ikf}{2} E_{0} e^{-ikf}
    \begin{bmatrix}
        \mathrm{I}_{02}\sin( 2\varphi )\\
        \mathrm{I}_{00} - \mathrm{I}_{02}\cos( 2\varphi )\\
        -2i\ \mathrm{I}_{01}\sin (\varphi) 
    \end{bmatrix}
    \equiv
    \frac{1}{Z_{\text{imp.}}}
    \cdot
    \varSigma \cdot \mathbf{I}^H_{(0, 0)}
$$

## Animate Bessel integrands and integrals

### Focal plane: $\mathrm{Re}[\mathrm{I}_{0n}]$ vs. $\rho$

In the focal plane $z=0$, only the real parts of the integrands of $\mathrm{I}_{00}, \mathrm{I}_{01}, \mathrm{I}_{02}$ are non-zero, since the imaginary parts all stem from $e^{ikz\cos\theta}$

In [ ]:
mosaic = [
    ['integrand'],
    ['integral'],
]
fig = plt.figure(figsize=(FS, 3.5 * len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

fig.suptitle(
    r'All imaginary parts of the integrands are zero in the focal plane $z=0$'
    '\n'
    r'(since they stem purely from $e^{ikz\cos\theta}$)'
)

optics = FocussingOptics()
λ = optics.λ
ρ_lim, ρ_step = 4, 0.01
RHO_VALS = np.arange(0, ρ_lim + ρ_step, ρ_step) * λ
label_text = r'$ \ \mathregular{integrand}(\rho, 0, \theta)]$'

Re_I00_points = np.real(optics.I00(RHO_VALS, 0))
Re_I01_points = np.real(optics.I01(RHO_VALS, 0))
Re_I02_points = np.real(optics.I02(RHO_VALS, 0))

step = 0.05*degrees
θ_range = np.arange(0, optics.θ_max + step, step)

frames = []
for ii, ρ_val in preview_frames(enumerate(RHO_VALS)):
    frame = []

    ax = ax_dict['integrand']
    _ = ax.text(5, 0.75, r'$\rho =$' + f'{ρ_val / λ:.2f}' + r'$\lambda$', color=grey, fontsize=16)
    frame.append(_)
    _ = ax.fill_between(θ_range / degrees, [np.real(optics.I0n_integrand(0, ρ_val, 0, θ)) for θ in θ_range], label='Re' + r'$[\mathrm{I}_{00}$' + label_text, alpha=0.5, color=blue)
    frame.append(_)
    _ = ax.fill_between(θ_range / degrees, [np.real(optics.I0n_integrand(1, ρ_val, 0, θ)) for θ in θ_range], label='Re' + r'$[\mathrm{I}_{01}$' + label_text, alpha=0.5, color=orange)
    frame.append(_)
    _ = ax.fill_between(θ_range / degrees, [np.real(optics.I0n_integrand(2, ρ_val, 0, θ)) for θ in θ_range], label='Re' + r'$[\mathrm{I}_{02}$' + label_text, alpha=0.5, color=green)
    frame.append(_)

    ax = ax_dict['integral']
    _, = ax.plot(RHO_VALS[:ii + 1] / λ, Re_I00_points[:ii + 1], lw=2, alpha=0.8, label='Re' + r'$[\mathrm{I}_{00}(\rho, 0)]$', color=blue)
    frame.append(_)
    _, = ax.plot(RHO_VALS[:ii + 1] / λ, Re_I01_points[:ii + 1], lw=2, alpha=0.8, label='Re' + r'$[\mathrm{I}_{01}(\rho, 0)]$', color=orange)
    frame.append(_)
    _, = ax.plot(RHO_VALS[:ii + 1] / λ, Re_I02_points[:ii + 1], lw=2, alpha=0.8, label='Re' + r'$[\mathrm{I}_{02}(\rho, 0)]$', color=green)
    frame.append(_)

    if ρ_val == RHO_VALS[0]:
        for ax in ax_dict.values():
            ax.legend()

    frames.append(frame)

text_box_str = (
    r'$\mathrm{NA} = $' + f'{optics.NA:.2f}'
    '\n'
    r'$\theta_{\mathregular{max}} = $' + f'{optics.θ_max / degrees:.2f}°'
    '\n'
    r'$w_0 = $' + f'{optics.w0 / mm:.1f} mm'
    '\n'
    r'$f = $' + f'{optics.focal_length / mm:.3f} mm'
    '\n'
    r'$a = $' + f'{optics.aperture_radius / mm:.3f} mm'
    '\n'
    r'$f_0 = $' + f'{optics.f0:.3f}'
)

ax = ax_dict['integrand']
ax.set_title('Integrands (integrals are the fill area)')
θ_max_deg = optics.θ_max / degrees
ax.text(1.05 * θ_max_deg, -0.15, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
ax.set_xlim(0, 1.4 * θ_max_deg)
ax.set_ylim(-0.5, 1)
ax.set_xlabel(r'Angle of (parallel) incoming ray to lens surface, $\theta$ [°]')
ax.vlines(θ_max_deg, -1.5, 1.5, ls='--', lw=0.9, color=grey)

ax = ax_dict['integral']
ax.set_title(r'Integrals (area of the integrands up to $\theta_{\mathregular{max}}$)')
ax.text(0.4*ρ_lim, 0.3, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
ax.set_xlabel(r'$\rho$ [$\lambda$]')
ax.set_xlim(0, ρ_lim)
ax.set_ylim(-0.2, 0.65)
ax.hlines(0, -0.1, 1.1*ρ_lim, alpha=0.5, ls='--', lw=1.5, color=grey)
ax.plot(RHO_VALS / λ, Re_I00_points, alpha=0.3, lw=4, color=blue)
ax.plot(RHO_VALS / λ, Re_I01_points, alpha=0.3, lw=4, color=orange)
ax.plot(RHO_VALS / λ, Re_I02_points, alpha=0.3, lw=4, color=green)

ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'Focal plane Re I_0n vs. rho')


### Propagation axis: $\mathrm{I}_{00}$ vs. $z$

Only the real and imaginary parts $\mathrm{I}_{00}$'s integrand are non-zero along the $z$-axis, since $\rho=0 \Rightarrow \kappa = 0 \Rightarrow J_{n\geq1}(\kappa)=0$ i.e. the Bessel functions for $n\geq1$ are all zero when their argument is zero.

In [ ]:
mosaic = [
    ['integrand'],
    ['integral'],
]
fig = plt.figure(figsize=(FS, 3.5 * len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

fig.suptitle(
    r'Only $\mathrm{I}_{00}$ is non-zero along the $z$-axis'
    '\n'
    r'(since $\rho=0 \Rightarrow \kappa=0 \Rightarrow J_{n\geq1}(0) = 0$)'
)

optics = FocussingOptics()
λ = optics.λ
z_lim, z_step = 3, 0.01
Z_VALS = np.arange(-z_lim, z_lim + z_step, z_step) * λ
label_text = r'$ \ \mathregular{integrand}(0, z, \theta)]$'

Re_I00_points = np.real(optics.I00(0, Z_VALS))
Im_I00_points = np.imag(optics.I00(0, Z_VALS))

step = 0.05*degrees
θ_range = np.arange(0, optics.θ_max + step, step)

frames = []
for ii, z_val in preview_frames(enumerate(Z_VALS)):
    frame = []

    ax = ax_dict['integrand']
    _ = ax.text(5, 0.85, r'$z =$' + f'{z_val / λ:.2f}' + r'$\lambda$', color=grey, fontsize=16)
    frame.append(_)
    _ = ax.fill_between(θ_range / degrees, [np.real(optics.I0n_integrand(0, 0, z_val, θ)) for θ in θ_range], label='Re' + r'$[\mathrm{I}_{00}$' + label_text, alpha=0.5, color=blue)
    frame.append(_)
    _ = ax.fill_between(θ_range / degrees, [np.imag(optics.I0n_integrand(0, 0, z_val, θ)) for θ in θ_range], label='Im' + r'$[\mathrm{I}_{00}$' + label_text, alpha=0.5, color=light_blue)
    frame.append(_)

    ax = ax_dict['integral']
    _, = ax.plot(Z_VALS[:ii + 1] / λ, Re_I00_points[:ii + 1], lw=2, alpha=0.8, label='Re' + r'$[\mathrm{I}_{00}(0, z)]$', color=blue)
    frame.append(_)
    _, = ax.plot(Z_VALS[:ii + 1] / λ, Im_I00_points[:ii + 1], lw=2, alpha=0.8, label='Im' + r'$[\mathrm{I}_{00}(0, z)]$', color=light_blue)
    frame.append(_)

    if z_val == Z_VALS[0]:
        ax_dict['integrand'].legend(loc='lower right')
        ax_dict['integral'].legend(loc='lower right')

    frames.append(frame)

text_box_str = (
    r'$\mathrm{NA} = $' + f'{optics.NA:.2f}'
    '\n'
    r'$\theta_{\mathregular{max}} = $' + f'{optics.θ_max / degrees:.2f}°'
    '\n'
    r'$w_0 = $' + f'{optics.w0 / mm:.1f} mm'
    '\n'
    r'$f = $' + f'{optics.focal_length / mm:.3f} mm'
    '\n'
    r'$a = $' + f'{optics.aperture_radius / mm:.3f} mm'
    '\n'
    r'$f_0 = $' + f'{optics.f0:.3f}'
)

ax = ax_dict['integrand']
ax.set_title('Integrands (integrals are the fill area)')
θ_max_deg = optics.θ_max / degrees
ax.text(1.05 * θ_max_deg, 0, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
ax.set_xlim(0, 1.3 * θ_max_deg)
ax.set_ylim(-1.5, 1.5)
ax.set_xlabel(r'Angle of (parallel) incoming ray to lens surface, $\theta$ [°]')
ax.vlines(θ_max_deg, -1.5, 1.5, ls='--', lw=0.8, color=grey)

ax = ax_dict['integral']
ax.set_title(r'Integrals (area of the integrands up to $\theta_{\mathregular{max}}$)')
ax.text(0.6*z_lim, 0.20, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
ax.set_xlabel(r'$z$ [$\lambda$]')
ax.set_xlim(-z_lim, z_lim)
ax.set_ylim(-1, 1)
ax.hlines(0, -1.5*z_lim, 1.5*z_lim, alpha=0.5, ls='--', lw=0.8, color=grey)
ax.plot(Z_VALS / λ, Re_I00_points, alpha=0.3, lw=4, color=blue)
ax.plot(Z_VALS / λ, Im_I00_points, alpha=0.3, lw=4, color=light_blue)

ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'Propagation axis I_00 vs. z')


### Integrals vs. $f$

In [ ]:
mosaic = [
    ['Re[I0n(ρ, 0)]'],
    ['I00(0, z)'],
]
fig = plt.figure(figsize=(5, 3.5*len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

λ = FocussingOptics().λ
num_points = 150
λ_lim = 3
RHO = np.linspace(0, λ_lim, num_points) * λ
Z = np.linspace(-λ_lim, λ_lim, num_points) * λ

step = 0.1
f_lim = 10
FOCAL_LENGTHS = np.arange(0.1, f_lim + step, step) * mm

frames = []
for f in preview_frames(FOCAL_LENGTHS):
    frame = []

    optics = FocussingOptics(focal_length=f)
    text_box_str = (
        r'$\mathrm{NA} = $' + f'{optics.NA:.2f}'
        '\n'
        r'$w_0 = $' + f'{optics.w0 / mm:.1f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / mm:.1f} mm'
        '\n'
        r'$a = $' + f'{optics.aperture_radius / mm:.3f} mm'
        '\n'
        r'$f_0 = $' + f'{optics.f0:.3f}'
    )

    # NOTE: imaginary part is zero when z=0 since the only imaginary part is exp(ik·z·cosθ)
    ax = ax_dict['Re[I0n(ρ, 0)]']
    _ = ax.text(2.25, 0.1, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(RHO / λ, np.real(optics.I00(RHO, 0)), label=r'$\mathregular{Re}[\mathrm{I}_{00}(\rho, 0)]$', lw=2, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(RHO / λ, np.real(optics.I01(RHO, 0)), label=r'$\mathregular{Re}[\mathrm{I}_{01}(\rho, 0)]$', lw=1.5, alpha=0.8, color=orange)
    frame.append(_)
    _, = ax.plot(RHO / λ, np.real(optics.I02(RHO, 0)), label=r'$\mathregular{Re}[\mathrm{I}_{02}(\rho, 0)]$', lw=1.25, alpha=0.8, color=green)
    frame.append(_)

    # NOTE: only I00(0, z) is non-zero since Jn(0) = 0 for all n > 0
    ax = ax_dict['I00(0, z)']
    _ = ax.text(1.5, -0.9, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    I00_0z = optics.I00(0, Z)
    frame.append(_)
    _, = ax.plot(Z / λ, np.real(I00_0z), label=r'$\mathregular{Re}[\mathrm{I}_{00}(0, z)]$', lw=2, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(Z / λ, np.imag(I00_0z), label=r'$\mathregular{Im}[\mathrm{I}_{00}(0, z)]$', lw=2, alpha=0.8, color=light_blue)
    frame.append(_)

    if f == FOCAL_LENGTHS[0]:
        for ax in ax_dict.values():
            ax.legend(loc='upper right')

    frames.append(frame)

optics = FocussingOptics()
ax = ax_dict['Re[I0n(ρ, 0)]']
ax.set_title(r'$\mathregular{Re}[\mathrm{I}_{0n}(\rho, z=0)]$', fontsize=15)
ax.set_xlabel(r'$\rho \ \ [\lambda]$')
ax.set_xlim(0, λ_lim)
ax.set_ylim(-0.25, 1.05)
ax.plot(RHO / λ, np.real(optics.I00(RHO, 0)), label=r'$\mathregular{Re}[\mathrm{I}_{00}(\rho, 0)]$', lw=5, alpha=0.2, color=blue)
ax.plot(RHO / λ, np.real(optics.I01(RHO, 0)), label=r'$\mathregular{Re}[\mathrm{I}_{01}(\rho, 0)]$', lw=4, alpha=0.2, color=orange)
ax.plot(RHO / λ, np.real(optics.I02(RHO, 0)), label=r'$\mathregular{Re}[\mathrm{I}_{02}(\rho, 0)]$', lw=3, alpha=0.2, color=green)
ax.hlines(0, 0, λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)

ax = ax_dict['I00(0, z)']
ax.set_title(r'$\mathrm{I}_{00}(\rho=0, z)$', fontsize=15)
ax.set_xlabel(r'$z \ \ [\lambda]$')
I00_0z = optics.I00(0, Z)
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-1.05, 1.05)
ax.plot(Z / λ, np.real(I00_0z), label=r'$\mathregular{Re}[\mathrm{I}_{00}(0, z)]$', lw=5, alpha=0.2, color=blue)
ax.plot(Z / λ, np.imag(I00_0z), label=r'$\mathregular{Im}[\mathrm{I}_{00}(0, z)]$', lw=5, alpha=0.2, color=light_blue)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines([-3, -2, -1, 0, 1, 2, 3][3], -1.1, 1.1, ls='--', lw=0.8, alpha=0.5, color=grey)

ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'Integrals vs. f')


### Integrals vs.$\mathrm{NA}$

In [ ]:
mosaic = [
    ['Re[I0n(ρ, 0)]'],
    ['I00(0, z)'],
]
fig = plt.figure(figsize=(5, 3.5*len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

λ = FocussingOptics().λ
num_points = 150
λ_lim = 3
RHO = np.linspace(0, λ_lim, num_points) * λ
Z = np.linspace(-λ_lim, λ_lim, num_points) * λ

step = 0.01
NAs = np.arange(0.4, 1.00 + step, step)

frames = []
for NA in preview_frames(NAs):
    frame = []

    optics = FocussingOptics(NA=NA)
    text_box_str = (
        r'$\mathrm{NA} = $' + f'{optics.NA:.2f}'
        '\n'
        r'$\theta_{\mathregular{max}} = $' + f'{optics.θ_max / degrees:.2f}°'
        '\n'
        r'$w_0 = $' + f'{optics.w0 / mm:.1f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / mm:.3f} mm'
        '\n'
        r'$a = $' + f'{optics.aperture_radius / mm:.3f} mm'
        '\n'
        r'$f_0 = $' + f'{optics.f0:.3f}'
    )

    # NOTE: imaginary part is zero when z=0 since the only imaginary part is exp(ik·z·cosθ)
    ax = ax_dict['Re[I0n(ρ, 0)]']
    _ = ax.text(2.25, 0.1, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(RHO / λ, np.real(optics.I00(RHO, 0)), label=r'$\mathregular{Re}[\mathrm{I}_{00}(\rho, 0)]$', lw=2, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(RHO / λ, np.real(optics.I01(RHO, 0)), label=r'$\mathregular{Re}[\mathrm{I}_{01}(\rho, 0)]$', lw=1.5, alpha=0.8, color=orange)
    frame.append(_)
    _, = ax.plot(RHO / λ, np.real(optics.I02(RHO, 0)), label=r'$\mathregular{Re}[\mathrm{I}_{02}(\rho, 0)]$', lw=1.25, alpha=0.8, color=green)
    frame.append(_)

    # NOTE: only I00(0, z) is non-zero since Jn(0) = 0 for all n > 0
    ax = ax_dict['I00(0, z)']
    _ = ax.text(1.8, -0.9, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    I00_0z = optics.I00(0, Z)
    _, = ax.plot(Z / λ, np.real(I00_0z), label=r'$\mathregular{Re}[\mathrm{I}_{00}(0, z)]$', lw=2, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(Z / λ, np.imag(I00_0z), label=r'$\mathregular{Im}[\mathrm{I}_{00}(0, z)]$', lw=2, alpha=0.8, color=light_blue)
    frame.append(_)

    if NA == NAs[0]:
        for ax in ax_dict.values():
            ax.legend()

    frames.append(frame)

optics = FocussingOptics()
ax = ax_dict['Re[I0n(ρ, 0)]']
ax.set_title(r'$\mathregular{Re}[\mathrm{I}_{0n}(\rho, z=0)]$', fontsize=15)
ax.set_xlabel(r'$\rho \ \ [\lambda]$')
ax.set_xlim(0, λ_lim)
ax.set_ylim(-0.2, 1.05)
ax.plot(RHO / λ, np.real(optics.I00(RHO, 0)), label=r'$\mathregular{Re}[\mathrm{I}_{00}(\rho, 0)]$', lw=5, alpha=0.2, color=blue)
ax.plot(RHO / λ, np.real(optics.I01(RHO, 0)), label=r'$\mathregular{Re}[\mathrm{I}_{01}(\rho, 0)]$', lw=4, alpha=0.2, color=orange)
ax.plot(RHO / λ, np.real(optics.I02(RHO, 0)), label=r'$\mathregular{Re}[\mathrm{I}_{02}(\rho, 0)]$', lw=3, alpha=0.2, color=green)
ax.hlines(0, 0, λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)

ax = ax_dict['I00(0, z)']
ax.set_title(r'$\mathrm{I}_{00}(\rho=0, z)$', fontsize=15)
ax.set_xlabel(r'$z \ \ [\lambda]$')
I00_0z = optics.I00(0, Z)
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-1.05, 1.05)
ax.plot(Z / λ, np.real(I00_0z), label=r'$\mathregular{Re}[\mathrm{I}_{00}(0, z)]$', lw=5, alpha=0.2, color=blue)
ax.plot(Z / λ, np.imag(I00_0z), label=r'$\mathregular{Im}[\mathrm{I}_{00}(0, z)]$', lw=5, alpha=0.2, color=light_blue)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines([-3, -2, -1, 0, 1, 2, 3][3], -1.2, 1.2, ls='--', lw=0.8, alpha=0.5, color=grey)

ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'Integrals vs.NA')


## Complex scaling factor $\varSigma \equiv \left(\frac{-ikf}{2} E_0 e^{-ikf}\right)$ vs. $f$ and $w_0$

In [ ]:
mosaic = [
    ['f'],
    ['w0'],
]
fig = plt.figure(figsize=(FS, 3.5*len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

λ = FocussingOptics().λ
f_start, f_lim, f_step = 2935, 2936, 1/100
focal_lengths = np.arange(f_start, f_lim + f_step, f_step) * λ

w0_start, w0_lim, w0_step = 0.5, 5, 0.05
w0s = np.arange(w0_start, w0_lim + w0_step, w0_step) * mm

E_unit = Volt / μm
Σ_vs_f = [FocussingOptics(focal_length=f).Σ / E_unit for f in focal_lengths]

frames = []
for ii, f in preview_frames(enumerate(focal_lengths)):
    frame = []

    ax = ax_dict['f']
    _, = ax.plot(focal_lengths[:ii + 1] / λ, np.abs(Σ_vs_f[:ii + 1]), label=r'$|\Sigma|$', lw=2.00, color=green)
    frame.append(_)
    _, = ax.plot(focal_lengths[:ii + 1] / λ, np.real(Σ_vs_f[:ii + 1]), label=r'Re[$\Sigma$]', lw=1.50, color=blue)
    frame.append(_)
    _, = ax.plot(focal_lengths[:ii + 1] / λ, np.imag(Σ_vs_f[:ii + 1]), label=r'Im[$\Sigma$]', lw=1.25, color=orange)
    frame.append(_)

    ax = ax_dict['w0']
    Σ_vs_w0 = np.array([FocussingOptics(w0=w0, focal_length=f).Σ / E_unit for w0 in w0s])
    _ = ax.text(1.3, 200, r'$f =$' + f'{f / λ:.2f}' + r'$\lambda = $' + f'{f / mm:.4f} mm', color=grey, fontsize=14)
    frame.append(_)
    _ = ax.text(2, 120, r'Arg[$\Sigma$]$=$' + f'{np.angle(Σ_vs_w0[0]) / degrees:.0f}°', color=grey, fontsize=14)
    frame.append(_)
    _, = ax.plot(w0s / mm, np.real(Σ_vs_w0), label=r'Re[$\Sigma$]', lw=2.00, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(w0s / mm, np.imag(Σ_vs_w0), label=r'Im[$\Sigma$]', lw=1.50, alpha=0.8, color=orange)
    frame.append(_)

    if f == focal_lengths[0]:
        ax = ax_dict['w0']
        ax.plot(w0s / mm, np.abs(Σ_vs_w0), label=r'$|\Sigma|$', lw=3.00, alpha=0.3, color=green)
        for ax in ax_dict.values():
            ax.legend(loc='lower right')

    frames.append(frame)

ax = ax_dict['f']
ax.set_xlim(f_start, f_lim)
ax.set_title('Complex parts of ' + r'$\Sigma \equiv \left(\frac{-ikf}{2} E_0 e^{-ikf}\right)$ vs. $f$')
ax.set_xlabel(r'Focal length, $f$ [$\lambda$]')
ax.set_ylabel(r'[$\mathregular{V/\mu m}$]')
ax.plot(focal_lengths / λ, np.abs(Σ_vs_f), lw=6, alpha=0.3, color=green)
ax.plot(focal_lengths / λ, np.real(Σ_vs_f), lw=5, alpha=0.3, color=blue)
ax.plot(focal_lengths / λ, np.imag(Σ_vs_f), lw=4, alpha=0.3, color=orange)

ax = ax_dict['w0']
ax.set_xlim(w0_start, w0_lim)
ax.set_title('Complex parts of ' + r'$\Sigma \equiv \left(\frac{-ikf}{2} E_0 e^{-ikf}\right)$ vs. $w_0$')
ax.set_xlabel(r'Incoming Gaussian beam width, $w_0$ [mm]')
ax.set_ylabel(r'[$\mathregular{V/\mu m}$]')
E0_w0_str = r'$E_0 = \sqrt{\frac{4P_{\mathregular{laser}}}{\pi w_0^2 \varepsilon_0 c}} \propto \frac{1}{w_0}$'
ax.text(1.6, -200, E0_w0_str, color=grey, fontsize=14)

ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'Complex scaling factor Sigma')


## Animate vector part $\mathbf{I}^E_{(0, 0)}$

$$
\mathbf{I}^E_{(0, 0), x}(\rho, \varphi, z) \equiv \mathrm{I}_{00}(\rho, z) + \mathrm{I}_{02}(\rho, z)\cos(2\varphi)
\\
\mathbf{I}^E_{(0, 0), y}(\rho, \varphi, z) \equiv \mathrm{I}_{02}(\rho, z)\sin(2\varphi)
\\
\mathbf{I}^E_{(0, 0), z}(\rho, \varphi, z) \equiv -2i\mathrm{I}_{01}(\rho, z) \cos(\varphi)
$$

In the focal plane ($z=0$), only the real parts of $\mathrm{I}_{0n}$ are non-zero.

A point along the $x$-axis implies $\varphi=0$ therefore the only non-zero components are those with $\cos(0) = 1$

$$
\mathbf{I}^E_{(0, 0), x}(\rho, 0, 0) = \text{Re}[\mathrm{I}_{00}] + \text{Re}[\mathrm{I}_{02}]
\\
\mathbf{I}^E_{(0, 0), z}(\rho, 0, 0) = -2i \text{Re}[\mathrm{I}_{01}]
$$
i.e. the real part of the $x$-component and the imaginary part of the $z$-component.

A point along the $y$-axis implies $\varphi=\pi/2$, therefore the only non-zero component is
$$
\mathbf{I}^E_{(0, 0), x}(\rho, \tfrac{\pi}{2}, 0) = \text{Re}[\mathrm{I}_{00}] - \text{Re}[\mathrm{I}_{02}]
$$
i.e. just the real part of the $x$-component again
since $\cos(\pi) = -1$, and $\sin(\pi) = \cos(\tfrac{\pi}{2}) = 0$

A point along the $z$-axis therefore only has a non-zero $x$-component

$$
\mathbf{I}^E_{(0, 0), x}(0, 0, z) = \text{Re}[\mathrm{I}_{00}] + i\text{Im}[\mathrm{I}_{00}]
$$


### varying $f$

In [ ]:
mosaic = [['xy'], ['z']]
fig = plt.figure(figsize=(FS, 3.5*len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})
fig.suptitle(r'Non-zero complex components of $\mathbf{I}^E_{(0, 0)}$ vs. $f$')

λ = FocussingOptics().λ
λ_lim, λ_step = 2, 1/50
VALS = np.arange(-λ_lim, λ_lim + λ_step, λ_step) * λ

f_unit, f_start, f_range, f_step = mm, 0.1, 10, 0.1
FOCAL_LENGTHS = np.arange(f_start, f_start + f_range + f_step, f_step) * f_unit

frames = []
for f in preview_frames(FOCAL_LENGTHS):
    frame = []

    optics = FocussingOptics(focal_length=f)
    text_box_str = (
        r'$\mathrm{NA} = $' + f'{optics.NA:.2f}'
        '\n'
        r'$w_0 = $' + f'{optics.w0 / mm:.1f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / mm:.1f} mm'
        '\n'
        r'$a = $' + f'{optics.aperture_radius / mm:.3f} mm'
        '\n'
        r'$f_0 = $' + f'{optics.f0:.2f}'
    )

    ax = ax_dict['xy']
    IE_00_x_vs_x = np.array(optics.IE_00('x', VALS, 0, 0))
    IE_00_x_vs_y = np.array(optics.IE_00('x', 0, VALS, 0))
    IE_00_z_vs_x = np.array(optics.IE_00('z', VALS, 0, 0))
    _ = ax.text(-0.8*λ_lim, 0.3, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(VALS / λ, np.real(IE_00_x_vs_x), label=r'$\mathregular{Re}[\mathbf{I}^E_{(0, 0), x}]$ vs. $x$', lw=2.00, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(VALS / λ, np.real(IE_00_x_vs_y), label=r'$\mathregular{Re}[\mathbf{I}^E_{(0, 0), x}]$ vs. $y$', lw=1.50, alpha=0.8, color=orange)
    frame.append(_)
    _, = ax.plot(VALS / λ, np.imag(IE_00_z_vs_x), label=r'$\mathregular{Im}[\mathbf{I}^E_{(0, 0), z}]$ vs. $x$', lw=1.25, alpha=0.8, color=green)
    frame.append(_)

    ax = ax_dict['z']
    IE_00_x_vs_z = np.array(optics.IE_00('x', 0, 0, VALS))
    _ = ax.text(0.6*λ_lim, -0.5, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(VALS / λ, np.real(IE_00_x_vs_z), label=r'$\mathregular{Re}[\mathbf{I}^E_{(0, 0), x}]$ vs. $z$', lw=2.00, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(VALS / λ, np.imag(IE_00_x_vs_z), label=r'$\mathregular{Im}[\mathbf{I}^E_{(0, 0), x}]$ vs. $z$', lw=1.25, alpha=0.8, color=light_blue)
    frame.append(_)

    if f == FOCAL_LENGTHS[0]:
        for ax in ax_dict.values():
            ax.legend(loc='upper right')

    frames.append(frame)

optics = FocussingOptics()

ax = ax_dict['xy']
ax.set_xlabel(r'$x, y$ [$\lambda$]')
IE_00_x_vs_x = np.array(optics.IE_00('x', VALS, 0, 0))
IE_00_x_vs_y = np.array(optics.IE_00('x', 0, VALS, 0))
IE_00_z_vs_x = np.array(optics.IE_00('z', VALS, 0, 0))
ax.plot(VALS / λ, np.real(IE_00_x_vs_x), lw=5, alpha=0.2, color=blue)
ax.plot(VALS / λ, np.real(IE_00_x_vs_y), lw=4, alpha=0.2, color=orange)
ax.plot(VALS / λ, np.imag(IE_00_z_vs_x), lw=3, alpha=0.2, color=green)
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-0.25, 0.65)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines(0, -0.20, 0.70, ls='--', lw=0.8, alpha=0.5, color=grey)

ax = ax_dict['z']
ax.set_xlabel(r'$z$ [$\lambda$]')
IE_00_x_vs_z = np.array(optics.IE_00('x', 0, 0, VALS))
ax.plot(VALS / λ, np.real(IE_00_x_vs_z), lw=5, alpha=0.2, color=blue)
ax.plot(VALS / λ, np.imag(IE_00_x_vs_z), lw=3, alpha=0.2, color=light_blue)
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-0.65, 0.65)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines(0, -0.70, 0.70, ls='--', lw=0.8, alpha=0.5, color=grey)

ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'varying f')


### varying $\mathrm{NA}$

In [ ]:
mosaic = [['xy'], ['z']]
fig = plt.figure(figsize=(FS, 3.5*len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})
fig.suptitle(r'Non-zero complex components of $\mathbf{I}^E_{(0, 0)}$ vs. $\mathrm{NA}$')

λ = FocussingOptics().λ
λ_lim, λ_step = 2, 1/50
VALS = np.arange(-λ_lim, λ_lim + λ_step, λ_step) * λ
step = 0.01
NAs = np.arange(0.4, 1.0 + step, step)

frames = []
for NA in preview_frames(NAs):
    frame = []

    optics = FocussingOptics(NA=NA)
    text_box_str = (
        r'$\mathrm{NA} = $' + f'{optics.NA:.2f}'
        '\n'
        r'$\theta_{\mathregular{max}} = $' + f'{optics.θ_max / degrees:.2f}°'
        '\n'
        r'$w_0 = $' + f'{optics.w0 / mm:.1f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / mm:.3f} mm'
        '\n'
        r'$a = $' + f'{optics.aperture_radius / mm:.3f} mm'
        '\n'
        r'$f_0 = $' + f'{optics.f0:.2f}'
    )

    ax = ax_dict['xy']
    IE_00_x_vs_x = np.array(optics.IE_00('x', VALS, 0, 0))
    IE_00_x_vs_y = np.array(optics.IE_00('x', 0, VALS, 0))
    IE_00_z_vs_x = np.array(optics.IE_00('z', VALS, 0, 0))
    _ = ax.text(-0.8*λ_lim, 0.3, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(VALS / λ, np.real(IE_00_x_vs_x), label=r'$\mathregular{Re}[\mathbf{I}^E_{(0, 0), x}]$ vs. $x$', lw=2.00, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(VALS / λ, np.real(IE_00_x_vs_y), label=r'$\mathregular{Re}[\mathbf{I}^E_{(0, 0), x}]$ vs. $y$', lw=1.50, alpha=0.8, color=orange)
    frame.append(_)
    _, = ax.plot(VALS / λ, np.imag(IE_00_z_vs_x), label=r'$\mathregular{Im}[\mathbf{I}^E_{(0, 0), z}]$ vs. $x$', lw=1.25, alpha=0.8, color=green)
    frame.append(_)

    ax = ax_dict['z']
    IE_00_x_vs_z = np.array(optics.IE_00('x', 0, 0, VALS))
    _ = ax.text(-0.8*λ_lim, 0.15, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(VALS / λ, np.real(IE_00_x_vs_z), label=r'$\mathregular{Re}[\mathbf{I}^E_{(0, 0), x}]$ vs. $z$', lw=2.00, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(VALS / λ, np.imag(IE_00_x_vs_z), label=r'$\mathregular{Im}[\mathbf{I}^E_{(0, 0), x}]$ vs. $z$', lw=1.25, alpha=0.8, color=light_blue)
    frame.append(_)

    if NA == NAs[0]:
        for ax in ax_dict.values():
            ax.legend()

    frames.append(frame)

optics = FocussingOptics()

ax = ax_dict['xy']
ax.set_xlabel(r'$x, y$ [$\lambda$]')
IE_00_x_vs_x = np.array(optics.IE_00('x', VALS, 0, 0))
IE_00_x_vs_y = np.array(optics.IE_00('x', 0, VALS, 0))
IE_00_z_vs_x = np.array(optics.IE_00('z', VALS, 0, 0))
ax.plot(VALS / λ, np.real(IE_00_x_vs_x), lw=5, alpha=0.2, color=blue)
ax.plot(VALS / λ, np.real(IE_00_x_vs_y), lw=4, alpha=0.2, color=orange)
ax.plot(VALS / λ, np.imag(IE_00_z_vs_x), lw=3, alpha=0.2, color=green)
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-0.5, 1)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines(0, -0.6, 1.1, ls='--', lw=0.8, alpha=0.5, color=grey)

ax = ax_dict['z']
ax.set_xlabel(r'$z$ [$\lambda$]')
IE_00_x_vs_z = np.array(optics.IE_00('x', 0, 0, VALS))
ax.plot(VALS / λ, np.real(IE_00_x_vs_z), lw=5, alpha=0.2, color=blue)
ax.plot(VALS / λ, np.imag(IE_00_x_vs_z), lw=3, alpha=0.2, color=light_blue)
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-1, 1)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines(0, -1.1, 1.1, ls='--', lw=0.8, alpha=0.5, color=grey)

ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'varying NA')


## Animate $x,y,z$ components of $\mathbf{E}_{\text{focus}}$

Since each component of the physical electric field is just the real part

$$
E_{\text{focus}, j} \equiv \text{Re}[\Sigma \cdot \mathbf{I}_{(0,0), j} \cdot e^{-i\omega t}]
$$

the only non-zero components in the focal plane are

In the $x$-axis
$$
E_{\text{focus}, x}(\rho, 0 , 0)
= \text{Re}[
    \Sigma
    \cdot
    (
        \text{Re}[\mathrm{I}_{00}] + \text{Re}[\mathrm{I}_{02}]
    )
    \cdot
    e^{-i\omega t}
]
\\
E_{\text{focus}, z}(\rho, 0 , 0)
= \text{Re}[
    \Sigma
    \cdot
    (
        -2i\text{Re}[\mathrm{I}_{01}]
    )
    \cdot
    e^{-i\omega t}
]
$$

In the $y$-axis
$$
E_{\text{focus}, x}(\rho, \tfrac{\pi}{2} , 0)
= \text{Re}[
    \Sigma
    \cdot
    (
        \text{Re}[\mathrm{I}_{00}] - \text{Re}[\mathrm{I}_{02}]
    )
    \cdot
    e^{-i\omega t}
]
$$

In the $z$-axis
$$
E_{\text{focus}, x}(0, 0, z)
= \text{Re}[
    \Sigma
    \cdot
    (
        \text{Re}[\mathrm{I}_{00}] + i\text{Im}[\mathrm{I}_{00}]
    )
    \cdot
    e^{-i\omega t}
]
$$

### varying $f$

##### slow variation

In [ ]:
mosaic = [['xy'], ['z']]
fig = plt.figure(figsize=(FS, 3.5*len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

E_unit = Volt / μm
λ = FocussingOptics().λ
λ_lim, λ_step = 2, 1/50
VALS = np.arange(-λ_lim, λ_lim + λ_step, λ_step) * λ

slow = 1
fast = not slow
title_str = r'of the $x, y, z$ components of $\mathbf{E}_{\mathregular{focus}}$'
if slow:
    fig.suptitle(r'Slow variation vs. $f$' + '\n' + title_str)
    f_start, f_end, f_step, f_unit = 500.75, 4000.75, 100, λ
if fast:
    fig.suptitle(r'Fast oscillation over 1 wavelength of $f$' + '\n' + title_str)
    f_start, f_end, f_step, f_unit = 2935, 2936, 0.01, λ
FOCAL_LENGTHS = np.arange(f_start, f_end + f_step, f_step) * f_unit

frames = []
for f in preview_frames(FOCAL_LENGTHS):
    frame = []

    optics = FocussingOptics(focal_length=f)
    text_box_str = (
        r'$\mathrm{NA} = $' + f'{optics.NA:.2f}'
        '\n'
        r'$w_0 = $' + f'{optics.w0 / mm:.1f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / mm:.3f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / λ:.2f} λ'
        '\n'
        r'$a = $' + f'{optics.aperture_radius / mm:.3f} mm'
        '\n'
        r'$f_0 = $' + f'{optics.f0:.3f}'
    )

    Ex_vs_x = np.array(optics.E_focus('x', VALS, 0, 0, 0)) / E_unit
    Ex_vs_y = np.array(optics.E_focus('x', 0, VALS, 0, 0)) / E_unit
    Ex_vs_z = np.array(optics.E_focus('x', 0, 0, VALS, 0)) / E_unit
    if fast:
        Ez_vs_x = np.array(optics.E_focus('z', VALS, 0, 0, 0)) / E_unit


    mag_E_vs_x = np.array(optics.E_focus('magnitude', VALS, 0, 0, 0)) / E_unit
    mag_E_vs_y = np.array(optics.E_focus('magnitude', 0, VALS, 0, 0)) / E_unit
    mag_E_vs_z = np.array(optics.E_focus('magnitude', 0, 0, VALS, 0)) / E_unit

    ax = ax_dict['xy']
    _ = ax.text(-0.8*λ_lim, -20, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(VALS / λ, Ex_vs_x, label=r'$E_{\mathregular{focus}, x}$ vs. $x$', lw=2.00, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(VALS / λ, Ex_vs_y, label=r'$E_{\mathregular{focus}, x}$ vs. $y$', lw=1.50, alpha=0.8, color=orange)
    frame.append(_)
    if fast:
        _, = ax.plot(VALS / λ, Ez_vs_x, label=r'$E_{\mathregular{focus}, z}$ vs. $x$', lw=1.25, alpha=0.8, color=green)
        frame.append(_)

    _, = ax.plot(VALS / λ, mag_E_vs_x, label=r'$|\mathbf{E}_{\mathregular{focus}}|$ vs. $x$', lw=1.2, alpha=0.6, color=blue, ls='--')
    frame.append(_)
    _, = ax.plot(VALS / λ, mag_E_vs_y, label=r'$|\mathbf{E}_{\mathregular{focus}}|$ vs. $y$', lw=1.2, alpha=0.6, color=orange, ls='--')
    frame.append(_)

    ax = ax_dict['z']
    _ = ax.text(-0.8*λ_lim, -20, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(VALS / λ, Ex_vs_z, label=r'$E_{\mathregular{focus}, x}$ vs. $z$', lw=2.00, alpha=0.8, color=green)
    frame.append(_)
    _, = ax.plot(VALS / λ, mag_E_vs_z, label=r'$|\mathbf{E}_{\mathregular{focus}}|$ vs. $z$', lw=1.2, alpha=0.6, color=green, ls='--')
    frame.append(_)

    if f == FOCAL_LENGTHS[0]:
        for ax in ax_dict.values():
            ax.legend(loc='upper right')

    frames.append(frame)

optics = FocussingOptics()

ax = ax_dict['xy']
ax.set_xlabel(r'$x, y$ [$\lambda$]')
ax.set_ylabel(r'[$\mathregular{V/\mu m}$]')
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-25, 25)
if fast:
    ax.plot(VALS / λ, np.array(optics.E_focus('z', VALS, 0, 0, 0)) / E_unit, lw=3, alpha=0.2, color=green)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines(0, -30, 30, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.plot(VALS / λ, np.array(optics.E_focus('x', VALS, 0, 0, 0)) / E_unit, lw=5, alpha=0.2, color=blue)
ax.plot(VALS / λ, np.array(optics.E_focus('x', 0, VALS, 0, 0)) / E_unit, lw=4, alpha=0.2, color=orange)

ax = ax_dict['z']
ax.set_xlabel(r'$z$ [$\lambda$]')
ax.set_ylabel(r'[$\mathregular{V/\mu m}$]')
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-25, 25)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines(0, -30, 30, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.plot(VALS / λ, np.array(optics.E_focus('x', 0, 0, VALS, 0)) / E_unit, lw=5, alpha=0.2, color=green)

ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'slow variation')


##### fast oscillation

In [ ]:
mosaic = [['xy'], ['z']]
fig = plt.figure(figsize=(FS, 3.5*len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

E_unit = Volt / μm
λ = FocussingOptics().λ
λ_lim, λ_step = 2, 1/50
VALS = np.arange(-λ_lim, λ_lim + λ_step, λ_step) * λ

slow = 0
fast = not slow
title_str = r'of the $x, y, z$ components of $\mathbf{E}_{\mathregular{focus}}$'
if slow:
    fig.suptitle(r'Slow variation vs. $f$' + '\n' + title_str)
    f_start, f_end, f_step, f_unit = 500.75, 4000.75, 100, λ
if fast:
    fig.suptitle(r'Fast oscillation over 1 wavelength of $f$' + '\n' + title_str)
    f_start, f_end, f_step, f_unit = 2935, 2936, 0.01, λ
FOCAL_LENGTHS = np.arange(f_start, f_end + f_step, f_step) * f_unit

frames = []
for f in preview_frames(FOCAL_LENGTHS):
    frame = []

    optics = FocussingOptics(focal_length=f)
    text_box_str = (
        r'$\mathrm{NA} = $' + f'{optics.NA:.2f}'
        '\n'
        r'$w_0 = $' + f'{optics.w0 / mm:.1f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / mm:.3f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / λ:.2f} λ'
        '\n'
        r'$a = $' + f'{optics.aperture_radius / mm:.3f} mm'
        '\n'
        r'$f_0 = $' + f'{optics.f0:.3f}'
    )

    Ex_vs_x = np.array(optics.E_focus('x', VALS, 0, 0, 0)) / E_unit
    Ex_vs_y = np.array(optics.E_focus('x', 0, VALS, 0, 0)) / E_unit
    Ex_vs_z = np.array(optics.E_focus('x', 0, 0, VALS, 0)) / E_unit
    if fast:
        Ez_vs_x = np.array(optics.E_focus('z', VALS, 0, 0, 0)) / E_unit


    mag_E_vs_x = np.array(optics.E_focus('magnitude', VALS, 0, 0, 0)) / E_unit
    mag_E_vs_y = np.array(optics.E_focus('magnitude', 0, VALS, 0, 0)) / E_unit
    mag_E_vs_z = np.array(optics.E_focus('magnitude', 0, 0, VALS, 0)) / E_unit

    ax = ax_dict['xy']
    _ = ax.text(-0.8*λ_lim, -20, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(VALS / λ, Ex_vs_x, label=r'$E_{\mathregular{focus}, x}$ vs. $x$', lw=2.00, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(VALS / λ, Ex_vs_y, label=r'$E_{\mathregular{focus}, x}$ vs. $y$', lw=1.50, alpha=0.8, color=orange)
    frame.append(_)
    if fast:
        _, = ax.plot(VALS / λ, Ez_vs_x, label=r'$E_{\mathregular{focus}, z}$ vs. $x$', lw=1.25, alpha=0.8, color=green)
        frame.append(_)

    _, = ax.plot(VALS / λ, mag_E_vs_x, label=r'$|\mathbf{E}_{\mathregular{focus}}|$ vs. $x$', lw=1.2, alpha=0.6, color=blue, ls='--')
    frame.append(_)
    _, = ax.plot(VALS / λ, mag_E_vs_y, label=r'$|\mathbf{E}_{\mathregular{focus}}|$ vs. $y$', lw=1.2, alpha=0.6, color=orange, ls='--')
    frame.append(_)

    ax = ax_dict['z']
    _ = ax.text(-0.8*λ_lim, -20, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(VALS / λ, Ex_vs_z, label=r'$E_{\mathregular{focus}, x}$ vs. $z$', lw=2.00, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(VALS / λ, mag_E_vs_z, label=r'$|\mathbf{E}_{\mathregular{focus}}|$ vs. $z$', lw=1.2, alpha=0.6, color=light_blue, ls='--')
    frame.append(_)

    if f == FOCAL_LENGTHS[0]:
        for ax in ax_dict.values():
            ax.legend(loc='upper right')

    frames.append(frame)

optics = FocussingOptics()

ax = ax_dict['xy']
ax.set_xlabel(r'$x, y$ [$\lambda$]')
ax.set_ylabel(r'[$\mathregular{V/\mu m}$]')
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-25, 25)
if fast:
    ax.plot(VALS / λ, np.array(optics.E_focus('z', VALS, 0, 0, 0)) / E_unit, lw=3, alpha=0.2, color=green)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines(0, -30, 30, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.plot(VALS / λ, np.array(optics.E_focus('x', VALS, 0, 0, 0)) / E_unit, lw=5, alpha=0.2, color=blue)
ax.plot(VALS / λ, np.array(optics.E_focus('x', 0, VALS, 0, 0)) / E_unit, lw=4, alpha=0.2, color=orange)

ax = ax_dict['z']
ax.set_xlabel(r'$z$ [$\lambda$]')
ax.set_ylabel(r'[$\mathregular{V/\mu m}$]')
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-25, 25)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines(0, -30, 30, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.plot(VALS / λ, np.array(optics.E_focus('x', 0, 0, VALS, 0)) / E_unit, lw=5, alpha=0.2, color=light_blue)

ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'fast oscillation')


### varying $\mathrm{NA}$

In [ ]:
mosaic = [['xy'], ['z']]
fig = plt.figure(figsize=(FS, 3.5*len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

E_unit = Volt / μm
λ = FocussingOptics().λ
λ_lim, λ_step = 2, 1/50
VALS = np.arange(-λ_lim, λ_lim + λ_step, λ_step) * λ

fig.suptitle(
    r'Variation vs. $\mathrm{NA}$'
    '\n'
    r'of the $x, y, z$ components of $\mathbf{E}_{\mathregular{focus}}$'
)
step = 0.01
NAs = np.arange(0.4, 1.0 + step, step)

frames = []
for NA in preview_frames(NAs):
    frame = []

    optics = FocussingOptics(NA=NA)
    text_box_str = (
        r'$\mathrm{NA} = $' + f'{optics.NA:.2f}'
        '\n'
        r'$\theta_{\mathregular{max}} = $' + f'{optics.θ_max / degrees:.2f}°'
        '\n'
        r'$w_0 = $' + f'{optics.w0 / mm:.1f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / mm:.3f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / λ:.2f} λ'
        '\n'
        r'$a = $' + f'{optics.aperture_radius / mm:.3f} mm'
        '\n'
        r'$f_0 = $' + f'{optics.f0:.3f}'
    )

    Ex_vs_x = np.array(optics.E_focus('x', VALS, 0, 0, 0)) / E_unit
    Ex_vs_y = np.array(optics.E_focus('x', 0, VALS, 0, 0)) / E_unit
    Ex_vs_z = np.array(optics.E_focus('x', 0, 0, VALS, 0)) / E_unit
    if fast:
        Ez_vs_x = np.array(optics.E_focus('z', VALS, 0, 0, 0)) / E_unit


    mag_E_vs_x = np.array(optics.E_focus('magnitude', VALS, 0, 0, 0)) / E_unit
    mag_E_vs_y = np.array(optics.E_focus('magnitude', 0, VALS, 0, 0)) / E_unit
    mag_E_vs_z = np.array(optics.E_focus('magnitude', 0, 0, VALS, 0)) / E_unit

    ax = ax_dict['xy']
    _ = ax.text(-0.8*λ_lim, -22, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(VALS / λ, Ex_vs_x, label=r'$E_{\mathregular{focus}, x}$ vs. $x$', lw=2.00, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(VALS / λ, Ex_vs_y, label=r'$E_{\mathregular{focus}, x}$ vs. $y$', lw=1.50, alpha=0.8, color=orange)
    frame.append(_)
    _, = ax.plot(VALS / λ, mag_E_vs_x, label=r'$|\mathbf{E}_{\mathregular{focus}}|$ vs. $x$', lw=1.2, alpha=0.6, color=blue, ls='--')
    frame.append(_)
    _, = ax.plot(VALS / λ, mag_E_vs_y, label=r'$|\mathbf{E}_{\mathregular{focus}}|$ vs. $y$', lw=1.2, alpha=0.6, color=orange, ls='--')
    frame.append(_)

    ax = ax_dict['z']
    _ = ax.text(-0.8*λ_lim, -22, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(VALS / λ, Ex_vs_z, label=r'$E_{\mathregular{focus}, x}$ vs. $z$', lw=2.00, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(VALS / λ, mag_E_vs_z, label=r'$|\mathbf{E}_{\mathregular{focus}}|$ vs. $z$', lw=1.2, alpha=0.6, color=light_blue, ls='--')
    frame.append(_)

    if NA == NAs[0]:
        for ax in ax_dict.values():
            ax.legend(loc='upper right')

    frames.append(frame)

optics = FocussingOptics()

ax = ax_dict['xy']
ax.set_xlabel(r'$x, y$ [$\lambda$]')
ax.set_ylabel(r'[$\mathregular{V/\mu m}$]')
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-28, 28)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines(0, -30, 30, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.plot(VALS / λ, np.array(optics.E_focus('x', VALS, 0, 0, 0)) / E_unit, lw=5, alpha=0.2, color=blue)
ax.plot(VALS / λ, np.array(optics.E_focus('x', 0, VALS, 0, 0)) / E_unit, lw=4, alpha=0.2, color=orange)

ax = ax_dict['z']
ax.set_xlabel(r'$z$ [$\lambda$]')
ax.set_ylabel(r'[$\mathregular{V/\mu m}$]')
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-28, 28)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines(0, -30, 30, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.plot(VALS / λ, np.array(optics.E_focus('x', 0, 0, VALS, 0)) / E_unit, lw=5, alpha=0.2, color=light_blue)

ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'varying NA (2)')


### varying $t$

In [ ]:
mosaic = [['xy'], ['z']]
fig = plt.figure(figsize=(FS, 3.5*len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

E_unit = Volt / μm
optics = FocussingOptics()
λ = optics.λ
λ_lim, λ_step = 2, 1/50
VALS = np.arange(-λ_lim, λ_lim + λ_step, λ_step) * λ

fig.suptitle(
    r'Variation vs. $t$'
    '\n'
    r'of the $x, y, z$ components of $\mathbf{E}_{\mathregular{focus}}$'
)

T = optics.T
step = 0.1 * T
times = np.arange(0, 2*T + step, step)

frames = []
for t in preview_frames(times):
    frame = []

    optics = FocussingOptics()
    text_box_str = (
        r'$\mathrm{NA} = $' + f'{optics.NA:.2f}'
        '\n'
        r'$w_0 = $' + f'{optics.w0 / mm:.1f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / mm:.3f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / λ:.2f} λ'
        '\n'
        r'$a = $' + f'{optics.aperture_radius / mm:.3f} mm'
        '\n'
        r'$f_0 = $' + f'{optics.f0:.3f}'
        '\n'
        r'$t = $' + f'{t / T:.2f}' + r'$T$'
    )

    Ex_vs_x = np.array(optics.E_focus('x', VALS, 0, 0, t)) / E_unit
    Ex_vs_y = np.array(optics.E_focus('x', 0, VALS, 0, t)) / E_unit
    Ex_vs_z = np.array(optics.E_focus('x', 0, 0, VALS, t)) / E_unit
    Ez_vs_x = np.array(optics.E_focus('z', VALS, 0, 0, t)) / E_unit


    mag_E_vs_x = np.array(optics.E_focus('magnitude', VALS, 0, 0, t)) / E_unit
    mag_E_vs_y = np.array(optics.E_focus('magnitude', 0, VALS, 0, t)) / E_unit
    mag_E_vs_z = np.array(optics.E_focus('magnitude', 0, 0, VALS, t)) / E_unit

    ax = ax_dict['xy']
    _ = ax.text(-0.8*λ_lim, -22, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(VALS / λ, Ex_vs_x, label=r'$E_{\mathregular{focus}, x}$ vs. $x$', lw=2.00, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(VALS / λ, Ex_vs_y, label=r'$E_{\mathregular{focus}, x}$ vs. $y$', lw=1.50, alpha=0.8, color=orange)
    frame.append(_)
    _, = ax.plot(VALS / λ, Ez_vs_x, label=r'$E_{\mathregular{focus}, z}$ vs. $x$', lw=1.25, alpha=0.8, color=green)
    frame.append(_)
    _, = ax.plot(VALS / λ, mag_E_vs_x, label=r'$|\mathbf{E}_{\mathregular{focus}}|$ vs. $x$', lw=1.2, alpha=0.6, color=blue, ls='--')
    frame.append(_)
    _, = ax.plot(VALS / λ, mag_E_vs_y, label=r'$|\mathbf{E}_{\mathregular{focus}}|$ vs. $y$', lw=1.2, alpha=0.6, color=orange, ls='--')
    frame.append(_)

    ax = ax_dict['z']
    _ = ax.text(-0.8*λ_lim, -22, text_box_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(VALS / λ, Ex_vs_z, label=r'$E_{\mathregular{focus}, x}$ vs. $z$', lw=2.00, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(VALS / λ, mag_E_vs_z, label=r'$|\mathbf{E}_{\mathregular{focus}}|$ vs. $z$', lw=1.2, alpha=0.6, color=light_blue, ls='--')
    frame.append(_)

    if t == times[0]:
        for ax in ax_dict.values():
            ax.legend(loc='upper right')

    frames.append(frame)

optics = FocussingOptics()

ax = ax_dict['xy']
ax.set_xlabel(r'$x, y$ [$\lambda$]')
ax.set_ylabel(r'[$\mathregular{V/\mu m}$]')
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-28, 28)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines(0, -30, 30, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.plot(VALS / λ, np.array(optics.E_focus('x', VALS, 0, 0, 0)) / E_unit, lw=5, alpha=0.2, color=blue)
ax.plot(VALS / λ, np.array(optics.E_focus('x', 0, VALS, 0, 0)) / E_unit, lw=4, alpha=0.2, color=orange)

ax = ax_dict['z']
ax.set_xlabel(r'$z$ [$\lambda$]')
ax.set_ylabel(r'[$\mathregular{V/\mu m}$]')
ax.set_xlim(-λ_lim, λ_lim)
ax.set_ylim(-28, 28)
ax.hlines(0, -1.1*λ_lim, 1.1*λ_lim, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.vlines(0, -30, 30, ls='--', lw=0.8, alpha=0.5, color=grey)
ax.plot(VALS / λ, np.array(optics.E_focus('x', 0, 0, VALS, 0)) / E_unit, lw=5, alpha=0.2, color=light_blue)

ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'varying t')


## Animate beam widths via focal plane intensity

### varying $f$

In [ ]:
lim = 1.5
step = 0.05
X = np.arange(-lim, lim + step, step) * μm
Y = np.arange(-lim, lim + step, step) * μm

mosaic = [['I']]
fig = plt.figure(figsize=(FS, 3.5*len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

optics = FocussingOptics()

ax = ax_dict['I']
ax.set_xlabel(r'$x,y \ \ \mathregular{[\mu m]}$')
ax.set_title(r'$I_{\mathregular{focus}}(x,y,z=0)$ vs. $x,y$')
ax.set_ylabel(r'$I_{\mathregular{focus}}$ [$P_{\mathregular{laser}}/\mathregular{\mu m^2}$]')
ax.set_ylim(0, 1)
ax.set_xlim(-lim, lim)
I_unit = optics.Power / μm**2

λ = optics.λ
step = 500
FOCAL_LENGTHS = np.arange(500.75, 4000.75 + step, step) * λ

frames = []
for f in preview_frames(FOCAL_LENGTHS):
    frame = []

    optics = FocussingOptics(Power=500*mW, focal_length=f)
    optics_str = (
        r'Focussing optics'
        '\n'
        r'$P_{\mathregular{laser}} = $' + f'{optics.Power / mW:.1f} mW'
        '\n'
        r'$\mathrm{NA} = $' + f'{optics.NA:.2f}'
        '\n'
        r'$w_0 = $' + f'{optics.w0 / mm:.2f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / mm:.2f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / λ:.2f}' + r'$\lambda$'
        '\n'
        r'$f_0 = $' + f'{optics.f0:.2f}'
    )

    I_FOCUS_X = optics.I_focus(X, 0, 0, 0) / I_unit
    I_FOCUS_Y = optics.I_focus(0, Y, 0, 0) / I_unit

    # https://stackoverflow.com/questions/10582795/finding-the-full-width-half-maximum-of-a-peak#10583774
    x_spline = UnivariateSpline(X / μm, I_FOCUS_X - np.max(I_FOCUS_X) * np.exp(-2), s=0)
    y_spline = UnivariateSpline(Y / μm, I_FOCUS_Y - np.max(I_FOCUS_Y) * np.exp(-2), s=0)
    r1y, r2y = y_spline.roots()
    r1x, r2x = x_spline.roots()
    # NOTE: the beam widths are the radius (not diameter!)
    wx0 = np.abs(r1x - r2x) / 2
    wy0 = np.abs(r1y - r2y) / 2

    # NOTE: invert x and y since in the experiment we take y-polarised light
    trap = Trap(tweezer=Tweezer(Power=500*mW, wx0=wy0*μm, wy0=wx0*μm))
    trap_str = (
        r'Tweezer trap'
        '\n'
        r'$P_{\mathregular{tw}} = $' + f'{trap.tweezer.Power / mW:.1f} mW'
        '\n'
        r'$\Omega_x = $' + f'{trap.Ωx / kHz:.1f} kHz'
        '\n'
        r'$\Omega_y = $' + f'{trap.Ωy / kHz:.1f} kHz'
        '\n'
        r'$\Omega_z = $' + f'{trap.Ωz / kHz:.1f} kHz'
        '\n'
    )

    ax = ax_dict['I']
    _ = ax.text(-0.9*lim, 0.5, optics_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _ = ax.text(0.5*lim, 0.5, trap_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(X / μm, I_FOCUS_X, label=r'$x =$ pol. axis', lw=1.8, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(Y / μm, I_FOCUS_Y, label=r'$y$', lw=1.8, alpha=0.8, color=orange)
    frame.append(_)
    _ = ax.axvspan(r1x, r2x, facecolor=blue, alpha=0.25)
    frame.append(_)
    _ = ax.axvspan(r1y, r2y, facecolor=orange, alpha=0.25)
    frame.append(_)

    hline_I_val = np.max(I_FOCUS_X) * np.exp(-2)
    _ = ax.text(0.8, 0.02 + hline_I_val, r'$1/e^2 \times$peak', color=grey)
    frame.append(_)
    _ = ax.hlines(hline_I_val, -2, 2, color=grey, ls='--', lw=0.8, alpha=0.5)
    frame.append(_)

    _ = ax.text(0.05, 1.35, r'$w_{x_0} =$' + f'{wx0:.3f}' + r'$\ \mathregular{\mu m}$', fontsize=8, color=blue)
    frame.append(_)
    _ = ax.text(0.05, 1.15, r'$w_{y_0} =$' + f'{wy0:.3f}' + r'$\ \mathregular{\mu m}$', fontsize=8, color=orange)
    frame.append(_)

    _ = ax.annotate('', xytext=(-0.02, 1.3), xy=(wx0 + 0.03, 1.3), arrowprops=dict(arrowstyle='<->', color=blue))
    frame.append(_)
    _ = ax.annotate('', xytext=(-0.02, 1.1), xy=(wx0 + 0.03, 1.1), arrowprops=dict(arrowstyle='<->', color=orange))
    frame.append(_)

    if f == FOCAL_LENGTHS[0]:
        for ax in ax_dict.values():
            ax.legend(loc='upper left')

    frames.append(frame)

ax.set_ylim(0, 1.5)
ax.vlines(0, -0.5, 2, color=grey, ls='--', lw=0.8, alpha=0.5)
ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'varying f (2)')


### varying $\mathrm{NA}$

In [ ]:
lim = 1.5
step = 0.05
X = np.arange(-lim, lim + step, step) * μm
Y = np.arange(-lim, lim + step, step) * μm

mosaic = [['I']]
fig = plt.figure(figsize=(FS, 3.5*len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

optics = FocussingOptics()
λ = optics.λ

ax = ax_dict['I']
ax.set_xlabel(r'$x,y \ \ \mathregular{[\mu m]}$')
z = 0.00 * μm
ax.set_title(r'$I_{\mathregular{focus}}(x,y,z=0)$ vs. $x,y$')
ax.set_ylabel(r'$I_{\mathregular{focus}}$ [$P_{\mathregular{laser}}/\mathregular{\mu m^2}$]')
ax.set_ylim(0, 1)
ax.set_xlim(-lim, lim)
I_unit = optics.Power / μm**2

step = 0.05
NAs = np.arange(0.5, 1.0 + step, step)

frames = []
for NA in preview_frames(NAs):
    frame = []

    optics = FocussingOptics(Power=500*mW, NA=NA)
    optics_str = (
        r'Focussing optics'
        '\n'
        r'$P_{\mathregular{laser}} = $' + f'{optics.Power / mW:.1f} mW'
        '\n'
        r'$\mathrm{NA} = $' + f'{optics.NA:.2f}'
        '\n'
        r'$\theta_{\mathregular{max}} = $' + f'{optics.θ_max / degrees:.2f}°'
        '\n'
        r'$w_0 = $' + f'{optics.w0 / mm:.2f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / mm:.2f} mm'
        '\n'
        r'$f = $' + f'{optics.focal_length / λ:.2f}' + r'$\lambda$'
        '\n'
        r'$f_0 = $' + f'{optics.f0:.2f}'
    )

    I_FOCUS_X = optics.I_focus(X, 0, z, 0) / I_unit
    I_FOCUS_Y = optics.I_focus(0, Y, z, 0) / I_unit

    # https://stackoverflow.com/questions/10582795/finding-the-full-width-half-maximum-of-a-peak#10583774
    x_spline = UnivariateSpline(X / μm, I_FOCUS_X - np.max(I_FOCUS_X) * np.exp(-2), s=0)
    y_spline = UnivariateSpline(Y / μm, I_FOCUS_Y - np.max(I_FOCUS_Y) * np.exp(-2), s=0)
    r1y, r2y = y_spline.roots()
    r1x, r2x = x_spline.roots()
    # NOTE: the beam widths are the radius (not diameter!)
    wx0 = np.abs(r1x - r2x) / 2
    wy0 = np.abs(r1y - r2y) / 2

    # NOTE: invert x and y since in the experiment we take y-polarised light
    trap = Trap(tweezer=Tweezer(Power=500*mW, wx0=wy0*μm, wy0=wx0*μm))
    trap_str = (
        r'Tweezer trap'
        '\n'
        r'$P_{\mathregular{tw}} = $' + f'{trap.tweezer.Power / mW:.1f} mW'
        '\n'
        r'$z_{\mathregular{eq.}} = $' + f'{trap.z_eq / μm:.3f} μm'
        '\n'
        r'$\Omega_x = $' + f'{trap.Ωx / kHz:.1f} kHz'
        '\n'
        r'$\Omega_y = $' + f'{trap.Ωy / kHz:.1f} kHz'
        '\n'
        r'$\Omega_z = $' + f'{trap.Ωz / kHz:.1f} kHz'
        '\n'
    )

    ax = ax_dict['I']
    _ = ax.text(-0.9*lim, 0.6, optics_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _ = ax.text(0.5*lim, 0.6, trap_str, fontsize=8, color=grey, bbox=dict(facecolor=grey, alpha=0.2, boxstyle='round,pad=0.5'))
    frame.append(_)
    _, = ax.plot(X / μm, I_FOCUS_X, label=r'$x =$ pol. axis', lw=1.8, alpha=0.8, color=blue)
    frame.append(_)
    _, = ax.plot(Y / μm, I_FOCUS_Y, label=r'$y$', lw=1.8, alpha=0.8, color=orange)
    frame.append(_)
    _ = ax.axvspan(r1x, r2x, facecolor=blue, alpha=0.25)
    frame.append(_)
    _ = ax.axvspan(r1y, r2y, facecolor=orange, alpha=0.25)
    frame.append(_)

    hline_I_val = np.max(I_FOCUS_X) * np.exp(-2)
    _ = ax.text(0.8, 0.02 + hline_I_val, r'$1/e^2 \times$peak', color=grey)
    frame.append(_)
    _ = ax.hlines(hline_I_val, -2, 2, color=grey, ls='--', lw=0.8, alpha=0.5)
    frame.append(_)

    _ = ax.text(0.05, 1.75, r'$w_{x_0} =$' + f'{wx0:.3f}' + r'$\ \mathregular{\mu m}$', fontsize=8, color=blue)
    frame.append(_)
    _ = ax.text(0.05, 1.55, r'$w_{y_0} =$' + f'{wy0:.3f}' + r'$\ \mathregular{\mu m}$', fontsize=8, color=orange)
    frame.append(_)

    _ = ax.annotate('', xytext=(-0.02, 1.7), xy=(wx0 + 0.03, 1.7), arrowprops=dict(arrowstyle='<->', color=blue))
    frame.append(_)
    _ = ax.annotate('', xytext=(-0.02, 1.5), xy=(wx0 + 0.03, 1.5), arrowprops=dict(arrowstyle='<->', color=orange))
    frame.append(_)

    if NA == NAs[0]:
        for ax in ax_dict.values():
            ax.legend(loc='upper left')

    frames.append(frame)

ax.set_ylim(0, 2)
ax.vlines(0, -0.5, 2, color=grey, ls='--', lw=0.8, alpha=0.5)
ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'varying NA (3)')


## Calculate beam widths

### via field magnitude

In [ ]:
def plot_widths_via_E_field(optics):
    num_points = 200
    lim = 2
    X = np.linspace(-lim, lim, num_points) * μm
    Y = np.linspace(-lim, lim, num_points) * μm

    E_field_unit = Volt / μm
    t = 0.25 * optics.T
    t = 0.0 * optics.T
    z = Trap().z_eq
    z = 0
    E_FOCUS_X = optics.magnitude_E_focus(X, 0, z, t) / E_field_unit
    E_FOCUS_Y = optics.magnitude_E_focus(0, Y, z, t) / E_field_unit

    fig, ax = plt.subplots(figsize=(FS, 3), dpi=250)
    ax.set_xlim(-0.9*lim, 0.9*lim)
    ax.set_ylim(-0.9*lim, 0.9*lim)

    ax.plot(X / μm, E_FOCUS_X, label=r'$x =$ pol. axis', lw=1.8, alpha=0.8)
    ax.plot(Y / μm, E_FOCUS_Y, label=r'$y$', lw=1.8, alpha=0.8)
    ax.set_title((
        # r'$|\mathbf{E}_{\mathregular{focus}}(x,y,z=$' + f'{z / μm:.2f} μm' + r'$)|$ vs. $x,y$'
        r'$|\mathbf{E}_{\mathregular{focus}}(x,y,z=0)|$ vs. $x,y$'
        '\n'
        r'$w_0=$' + f'{optics.w0 / mm:.1f} mm'
        ', '
        r'$f=$' + f'{optics.focal_length / mm:.3f} mm'
    ))
    ax.set_xlabel(r'$x,y \ \ \mathregular{[\mu m]}$')
    ax.set_ylabel(r'$|\mathbf{E}_{\mathregular{focus}}| \ \mathregular{[V/\mu m]}$')

    max_EX, max_EY = np.max(E_FOCUS_X), np.max(E_FOCUS_Y)
    max_EX, max_EY = optics.magnitude_E_focus(0, 0, 0, t) / E_field_unit, optics.magnitude_E_focus(0, 0, 0, t) / E_field_unit
    ax.set_ylim(0, 1.1 * max_EX)

    hline_val = max_EX * np.exp(-1)
    ax.hlines(hline_val, -1, 1, color=grey, ls='--', lw=0.8)
    ax.text(1.05, 0.95 * hline_val, r'$(1/e) \times$' + 'peak', color=grey)
    ax.text(-0.75*lim, 0.95*max_EX, r'$f_0 =$' + f'{optics.f0:.3f}', color=grey)
    ax.text(-0.78*lim, 1.02*max_EX, r'$\mathrm{NA}=$' + f'{optics.NA:.3f}', color=grey)

    # https://stackoverflow.com/questions/10582795/finding-the-full-width-half-maximum-of-a-peak#10583774
    spline = UnivariateSpline(X / μm, (E_FOCUS_X - max_EX * np.exp(-1)), s=0)
    r1, r2 = spline.roots()
    wx0 = np.abs(r1 - r2) / 2
    ax.axvspan(r1, r2, facecolor=blue, alpha=0.3, label=r'$w_{x_0} =$' + f'{wx0:.3f}' + r'$\ \mathregular{\mu m}$')

    spline = UnivariateSpline(Y / μm, (E_FOCUS_Y - max_EY * np.exp(-1)), s=0)
    r1, r2 = spline.roots()
    wy0 = np.abs(r1 - r2) / 2
    ax.axvspan(r1, r2, facecolor=orange, alpha=0.3, label=r'$w_{y_0} =$' + f'{wy0:.3f}' + r'$\ \mathregular{\mu m}$')
    ax.legend()

    # lower left corner coordinates
    # (x0, y0, width, height)
    sub_ax = ax.inset_axes((0, 0.5, 0.33, 0.33))

    sub_ax.text(0, 0, 'spot\nellipse', fontsize=8, color=red, alpha=0.6, horizontalalignment='center')
    sub_ax.text(0, -0.5, 'ref. circle', fontsize=4, color=grey, alpha=0.5, horizontalalignment='center')

    sub_ax.add_patch(Circle(xy=(0, 0), radius=wy0, color=grey, alpha=0.5, ls='--', fill=None))
    sub_ax.add_patch(Ellipse(xy=(0, 0), width=2*wx0, height=2*wy0, color=red, alpha=0.5))

    # https://stackoverflow.com/questions/1639463/how-to-change-border-width-of-a-matplotlib-graph#20009598
    [_.set_linewidth(0.1) for _ in sub_ax.spines.values()]
    sub_ax.set_facecolor((0,0,0,0))

    sub_ax.set_xlim(-1.1*wx0, 1.1*wx0)
    sub_ax.set_ylim(-1.1*wx0, 1.1*wx0)
    sub_ax.set(aspect=1, xticklabels=[], yticklabels=[])
    sub_ax.tick_params(right=False, left=False, bottom=False)

    sub_ax.text(0, -1.3*wx0, r'$x$', color=black, fontsize=6)
    sub_ax.text(-1.35*wx0, 0, r'$y$', color=black, fontsize=6)

    plt.show()

plot_widths_via_E_field(FocussingOptics())


### via intensity

In [ ]:
def plot_widths_via_intensity(optics):
    num_points = 200
    lim = 2
    X = np.linspace(-lim, lim, num_points) * μm
    Y = np.linspace(-lim, lim, num_points) * μm

    z = Trap().z_eq
    z = 0

    INTENSITY_X = optics.I_focus(X, 0, z, 0)
    INTENSITY_Y = optics.I_focus(0, Y, z, 0)

    fig, ax = plt.subplots(figsize=(FS, 3), dpi=250)

    # NOTE: the curve shape is not dependent on power, which simply scales it
    intensity_unit = optics.Power / μm**2
    ax.set_ylabel(r'Focal plane intensity [$P_{\mathregular{laser}} \mathregular{/\mu m^2}$]')

    ax.set_xlim(-0.9*lim, 0.9*lim)
    ax.set_ylim(-0.9*lim, 0.9*lim)

    ax.plot(X / μm, INTENSITY_X / intensity_unit, label=r'$x =$ pol. axis', lw=1.8, alpha=0.8)
    ax.plot(Y / μm, INTENSITY_Y / intensity_unit, label=r'$y$', lw=1.8, alpha=0.8)
    ax.set_title(r'Focal plane intensity$ = \frac{1}{2} \varepsilon_0 |\mathbf{E}_{\mathrm{focus}}(x,y,z=0)|^2 c$ vs. $x,y$')
    ax.set_xlabel(r'$x,y \ \ \mathregular{[\mu m]}$')

    max_IX = np.max(INTENSITY_X)
    max_IY = np.max(INTENSITY_Y)
    ax.set_ylim(0, 1.1 * max_IX / intensity_unit)

    hline_val = max_IX * np.exp(-2) / intensity_unit
    ax.hlines(hline_val, -1, 1, color=grey, ls='--')
    ax.text(1.05, 0.92 * hline_val, r'$(1/e^2) \times$' + 'peak', color=grey)
    ax.text(-0.75*lim, 0.95*max_IX / intensity_unit, r'$f_0 =$' + f'{optics.f0:.3f}', color=grey)
    ax.text(-0.78*lim, 1.02*max_IX / intensity_unit, r'$\mathrm{NA}=$' + f'{optics.NA:.3f}', color=grey)

    # https://stackoverflow.com/questions/10582795/finding-the-full-width-half-maximum-of-a-peak#10583774
    spline = UnivariateSpline(X / μm, (INTENSITY_X - max_IX * np.exp(-2)) / intensity_unit, s=0)
    r1, r2 = spline.roots()
    wx0 = np.abs(r1 - r2) / 2
    ax.axvspan(r1, r2, facecolor=blue, alpha=0.3, label=r'$w_{x_0} =$' + f'{wx0:.3f}' + r'$\ \mathregular{\mu m}$')

    spline = UnivariateSpline(Y / μm, (INTENSITY_Y - max_IY * np.exp(-2)) / intensity_unit, s=0)
    r1, r2 = spline.roots()
    wy0 = np.abs(r1 - r2) / 2
    ax.axvspan(r1, r2, facecolor=orange, alpha=0.3, label=r'$w_{y_0} =$' + f'{wy0:.3f}' + r'$\ \mathregular{\mu m}$')
    ax.legend()

    # lower left corner coordinates
    # (x0, y0, width, height)
    sub_ax = ax.inset_axes((0, 0.5, 0.33, 0.33))

    sub_ax.text(0, 0, 'spot\nellipse', fontsize=8, color=red, alpha=0.6, horizontalalignment='center')
    sub_ax.text(0, -0.3, 'ref. circle', fontsize=4, color=grey, alpha=0.5, horizontalalignment='center')

    sub_ax.add_patch(Circle(xy=(0, 0), radius=wy0, color=grey, alpha=0.5, ls='--', fill=None))
    sub_ax.add_patch(Ellipse(xy=(0, 0), width=2*wx0, height=2*wy0, color=red, alpha=0.5))

    # https://stackoverflow.com/questions/1639463/how-to-change-border-width-of-a-matplotlib-graph#20009598
    [_.set_linewidth(0.1) for _ in sub_ax.spines.values()]
    sub_ax.set_facecolor((0,0,0,0))

    sub_ax.set_xlim(-1.1*wx0, 1.1*wx0)
    sub_ax.set_ylim(-1.1*wx0, 1.1*wx0)
    sub_ax.set(aspect=1, xticklabels=[], yticklabels=[])
    sub_ax.tick_params(right=False, left=False, bottom=False)

    sub_ax.text(0, -1.3*wx0, r'$x$', color=black, fontsize=6)
    sub_ax.text(-1.35*wx0, 0, r'$y$', color=black, fontsize=6)

    plt.show()

plot_widths_via_intensity(FocussingOptics())


### vs. NA

In [ ]:
all_wx0_vs_NA = []
all_wy0_vs_NA = []
all_f0_vs_NA = []

NA_step = 0.05
all_NA = np.arange(0.3, 1 + NA_step, NA_step)

num_points = 50
lim = 4
X = np.linspace(-lim, lim, num_points) * μm
Y = np.linspace(-lim, lim, num_points) * μm

intensity_unit = mW / μm**2
for NA in all_NA:
    new_optics = FocussingOptics(NA=NA, focal_length=10_000.75*λ)
    new_optics = FocussingOptics(NA=NA)

    all_f0_vs_NA.append(new_optics.f0)

    INTENSITY_X = new_optics.I_focus(X, 0, 0, 0)
    INTENSITY_Y = new_optics.I_focus(0, Y, 0, 0)

    max_IX = np.max(INTENSITY_X)
    max_IY = np.max(INTENSITY_Y)

    # https://stackoverflow.com/questions/10582795/finding-the-full-width-half-maximum-of-a-peak#10583774
    spline = UnivariateSpline(X / μm, (INTENSITY_X - max_IX * np.exp(-2)) / intensity_unit, s=0)
    r1, r2 = spline.roots()
    wx0 = np.abs(r1 - r2) / 2
    all_wx0_vs_NA.append(wx0)

    spline = UnivariateSpline(Y / μm, (INTENSITY_Y - max_IY * np.exp(-2)) / intensity_unit, s=0)
    r1, r2 = spline.roots()
    wy0 = np.abs(r1 - r2) / 2
    all_wy0_vs_NA.append(wy0)

all_wx0_vs_NA = np.array(all_wx0_vs_NA)
all_wy0_vs_NA = np.array(all_wy0_vs_NA)
all_f0_vs_NA = np.array(all_f0_vs_NA)

In [ ]:
fig, ax = plt.subplots(figsize=(FS, 3), dpi=250)
ax.set_title(r'Focussed beam widths in $x,y$ vs. Numerical Aperture')
ax.plot(all_NA, all_wx0_vs_NA, 'o-', label=r'$w_{x_0}$')
ax.plot(all_NA, all_wy0_vs_NA, 'o-', label=r'$w_{y_0}$')
ax.set_xlabel(r'Numerical Aperture, $\mathrm{NA}\equiv n \ \sin \theta_{\mathregular{max}}$')
ax.set_ylabel(r'$w_{x_0}, w_{y_0}$ [$\mathregular{\mu m}$]')

ax.text(0.3, 0.7, r'$f=$' + f'{new_optics.focal_length / mm:.3f}mm', color=grey)
ax.text(0.3, 0.6, r'$w_0=$' + f'{new_optics.w0 / mm:.1f}mm', color=grey)
# ax.text(0.62, 0.75, r'$f_0=$' + f'{[float(f"{f0:.2f}") for f0 in all_f0_vs_NA]}', color=grey)

ax.hlines(1.0679, 0.1, 1.1, color=blue, ls='--')
ax.hlines(0.9276, 0.1, 1.1, color=orange, ls='--')
ax.vlines(0.77, 0.3, 1.5, color=grey, alpha=0.5, ls='--')
ax.set_xlim(0.2, 1.05)
ax.set_ylim(0.3, 1.5)

ax.legend()
plt.show()

### vs. filling factor (via focal length)

In [ ]:
all_wx0_vs_f0 = []
all_wy0_vs_f0 = []
all_f0 = []
num_focal_length_points = 40
λ = FocussingOptics().λ
step = 1_000
FOCAL_LENGTHS = np.arange(500.75, 20_000.75 + step, step) * λ

num_points = 50
lim = 6
X = np.linspace(-lim, lim, num_points) * μm
Y = np.linspace(-lim, lim, num_points) * μm

intensity_unit = mW / μm**2
for focal_length in FOCAL_LENGTHS:
    new_optics = FocussingOptics(focal_length=focal_length)
    all_f0.append(new_optics.f0)

    # plot_widths_via_E_field(new_optics)
    # plot_widths_via_intensity(new_optics)

    INTENSITY_X = new_optics.I_focus(X, 0, 0, 0)
    INTENSITY_Y = new_optics.I_focus(0, Y, 0, 0)

    max_IX = np.max(INTENSITY_X)
    max_IY = np.max(INTENSITY_Y)

    # https://stackoverflow.com/questions/10582795/finding-the-full-width-half-maximum-of-a-peak#10583774
    spline = UnivariateSpline(X / μm, (INTENSITY_X - max_IX * np.exp(-2)) / intensity_unit, s=0)
    r1, r2 = spline.roots()
    wx0 = np.abs(r1 - r2) / 2
    all_wx0_vs_f0.append(wx0)

    spline = UnivariateSpline(Y / μm, (INTENSITY_Y - max_IY * np.exp(-2)) / intensity_unit, s=0)
    r1, r2 = spline.roots()
    wy0 = np.abs(r1 - r2) / 2
    all_wy0_vs_f0.append(wy0)

all_wx0_vs_f0 = np.array(all_wx0_vs_f0)
all_wy0_vs_f0 = np.array(all_wy0_vs_f0)
all_f0 = np.array(all_f0)

In [ ]:
fig, ax = plt.subplots(figsize=(FS, 3), dpi=250)
ax.set_title((
    r'Focussed beam widths in $x,y$ vs. focal length'
    '\n'
    r'$\mathrm{NA}=$' + f'{new_optics.NA:.2f}'
    ', '
    r'$w_0=$' + f'{new_optics.w0 / mm:.1f} mm'
))
ax.plot(FOCAL_LENGTHS / mm, all_wx0_vs_f0, 'o-', label=r'$w_{x_0}$')
ax.plot(FOCAL_LENGTHS / mm, all_wy0_vs_f0, 'o-', label=r'$w_{y_0}$')
ax.set_xlabel(r'Focal length, $f$ [mm]')
ax.set_ylabel(r'$w_{x_0}, w_{y_0}$ [$\mathregular{\mu m}$]')

max_f = max(FOCAL_LENGTHS / mm)
ax.hlines(1.0679, -0.1, max_f, color=blue, ls='--')
ax.hlines(0.9276, -0.1, max_f, color=orange, ls='--')
ax.set_xlim(0, max_f)

ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(FS, 3), dpi=250)
ax.set_title(r'Focussed beam widths in $x,y$ vs. filling factor')
ax.plot(all_f0, all_wx0_vs_f0, 'o-', label=r'$w_{x_0}$')
ax.plot(all_f0, all_wy0_vs_f0, 'o-', label=r'$w_{y_0}$')
ax.set_xlabel(r'Filling factor, $f_0 \equiv \frac{w_0}{f \ \sin \theta_\mathrm{max}}$')
ax.set_ylabel(r'$w_{x_0}, w_{y_0}$ [$\mathregular{\mu m}$]')

ax.text(1.4, 1.0, r'$\mathrm{NA}=$' + f'{new_optics.NA:.2f}', color=grey)
ax.text(1.4, 1.03, r'$w_0=$' + f'{new_optics.w0 / mm:.1f} mm', color=grey)

max_f0 = max(all_f0)
ax.hlines(1.0679, -0.1, max_f0, color=blue, ls='--')
ax.hlines(0.9276, -0.1, max_f0, color=orange, ls='--')
ax.set_xlim(0, max_f0)

ax.legend()
plt.show()

### vs. filling factor (via $w_0$)

In [ ]:
all_wx0_vs_f0 = []
all_wy0_vs_f0 = []
all_f0 = []
num_w0_points = 10

all_w0 = np.linspace(0.5, 5, num_w0_points) * mm

num_points = 50
lim = 4
X = np.linspace(-lim, lim, num_points) * μm
Y = np.linspace(-lim, lim, num_points) * μm

intensity_unit = mW / μm**2
for w0 in all_w0:
    new_optics = FocussingOptics(w0=w0)
    all_f0.append(new_optics.f0)

    INTENSITY_X = new_optics.I_focus(X, 0, 0, 0)
    INTENSITY_Y = new_optics.I_focus(0, Y, 0, 0)

    max_IX = np.max(INTENSITY_X)
    max_IY = np.max(INTENSITY_Y)

    # https://stackoverflow.com/questions/10582795/finding-the-full-width-half-maximum-of-a-peak#10583774
    spline = UnivariateSpline(X / μm, (INTENSITY_X - max_IX * np.exp(-2)) / intensity_unit, s=0)
    r1, r2 = spline.roots()
    wx0 = np.abs(r1 - r2) / 2
    all_wx0_vs_f0.append(wx0)

    spline = UnivariateSpline(Y / μm, (INTENSITY_Y - max_IY * np.exp(-2)) / intensity_unit, s=0)
    r1, r2 = spline.roots()
    wy0 = np.abs(r1 - r2) / 2
    all_wy0_vs_f0.append(wy0)

all_wx0_vs_f0 = np.array(all_wx0_vs_f0)
all_wy0_vs_f0 = np.array(all_wy0_vs_f0)
all_f0 = np.array(all_f0)

In [ ]:
fig, ax = plt.subplots(figsize=(FS, 3), dpi=250)
ax.set_title(r'Focussed beam widths in $x,y$ vs. filling factor')
ax.plot(all_f0, all_wx0_vs_f0, 'o-', label=r'$w_{x_0}$')
ax.plot(all_f0, all_wy0_vs_f0, 'o-', label=r'$w_{y_0}$')
ax.set_xlabel(r'Filling factor, $f_0 \equiv \frac{w_0}{f \ \sin \theta_\mathrm{max}}$')
ax.set_ylabel(r'$w_{x_0}, w_{y_0}$ [$\mathregular{\mu m}$]')

ax.text(1.4, 1.4, r'$\mathrm{NA}=$' + f'{new_optics.NA:.2f}', color=grey)

max_f0 = max(all_f0)
ax.hlines(1.0679, -0.1, max_f0, color=blue, ls='--')
ax.hlines(0.9276, -0.1, max_f0, color=orange, ls='--')
ax.set_xlim(0, max_f0)

ax.legend()
plt.show()


## Beam ellipticity

In [ ]:
# a = long axis
# b = short axis
a = all_wx0_vs_f0
b = all_wy0_vs_f0


# NOTE: default trap's ellipticity and flattening are
# ellipticity = 0.49548
# flattening = 0.13138
ellipticity = np.sqrt((a**2 - b**2)/a**2)
# flattening = 1 - np.sqrt(1 - ellipticity**2)

# https://stackoverflow.com/questions/25668828/how-to-create-colour-gradient-in-python#29736037
# https://bsouthga.dev/posts/color-gradients-with-python
colour_gradient = hsl_gradient(red, purple, len(all_f0))

fig, ax = plt.subplots(figsize=(FS, 3), dpi=250)
ax.set_title(r'Beam ellipticity$=\sqrt{\frac{w_{x_0}^2 - w_{y_0}^2}{w_{x_0}^2}}$ vs. $f_0$')
ax.scatter(
    all_f0, ellipticity,
    color=colour_gradient,
)
ax.set_xlabel(r'Filling factor, $f_0 \equiv \frac{w_0}{f \ \sin \theta_\mathrm{max}}$')

fig, ax = plt.subplots(figsize=(FS, FS), dpi=250)
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-0.1, 1.1)

def ellipse(n, scale_factor=0.1):
    scaled_x_coord = (all_f0[n] - np.min(all_f0)) / (np.max(all_f0) - np.min(all_f0))
    scaled_y_coord = (ellipticity[n] - np.min(ellipticity)) / (np.max(ellipticity) - np.min(ellipticity))
    sf = scale_factor
    return Ellipse(
        xy=(scaled_x_coord, scaled_y_coord),
        width=a[n] * sf,
        height=b[n] * sf,
        color=colour_gradient[n],
        alpha=0.5,
    )

def outer_circle(n, scale_factor=0.1):
    scaled_x_coord = (all_f0[n] - np.min(all_f0)) / (np.max(all_f0) - np.min(all_f0))
    scaled_y_coord = (ellipticity[n] - np.min(ellipticity)) / (np.max(ellipticity) - np.min(ellipticity))
    sf = scale_factor
    return Circle(
        xy=(scaled_x_coord, scaled_y_coord),
        radius=0.5 * a[n] * sf,
        color=grey,
        ls='--',
        lw=0.5,
        fill=None,
        alpha=0.3,
    )

def inner_circle(n, scale_factor=0.1):
    scaled_x_coord = (all_f0[n] - np.min(all_f0)) / (np.max(all_f0) - np.min(all_f0))
    scaled_y_coord = (ellipticity[n] - np.min(ellipticity)) / (np.max(ellipticity) - np.min(ellipticity))
    sf = scale_factor
    return Circle(
        xy=(scaled_x_coord, scaled_y_coord),
        radius=0.5 * b[n] * sf,
        color=grey,
        ls='--',
        lw=0.5,
        fill=None,
        alpha=0.3,
    )


for _ in range(8):
    ax.add_patch(ellipse(_))
    # ax.add_patch(outer_circle(_))
    ax.add_patch(inner_circle(_))


fig, ax = plt.subplots(figsize=(FS, 5), dpi=250)
ax.add_patch(Ellipse(xy=(0, 0), width=a[0], height=b[0], color=red, alpha=0.5, label=f'{ellipticity[0]:.3f}'))
ax.add_patch(Ellipse(xy=(0, 0), width=a[-1], height=b[-1], color=purple, alpha=0.5, label=f'{ellipticity[-1]:.3f}'))

ax.add_patch(Ellipse(xy=(0, 0), width=b[0], height=b[0], color=red, alpha=0.3, lw=0.5, ls='--', fill=None))
ax.add_patch(Ellipse(xy=(0, 0), width=b[-1], height=b[-1], color=purple, alpha=0.3, lw=0.5, ls='--', fill=None))


ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)

ax.legend()

plt.show()

## Contour plots of  $E_{j,\text{focus}}^2$ and $|\mathbf{E}_{\text{focus}}|^2$

NOTE: try the non-focal equilibrium z-plane $z_{\text{eq}} = \frac{\alpha' k \gamma_0}{\alpha''} = \frac{\mathrm{Re}[\alpha_{\text{eff}}]}{\mathrm{Im}[\alpha_{\text{eff}}]}k \gamma_0$ where $\gamma_0 \equiv z_{\text{R}}^2 - z_{\text{R}}/k$

In [ ]:
# NOTE: how to do the log scale and colourbar
# https://stackoverflow.com/questions/65823932/plt-contourf-with-given-number-of-levels-in-logscale

# NOTE: how to do the streamline plots
# https://matplotlib.org/stable/gallery/images_contours_and_fields/plot_streamplot.html#sphx-glr-gallery-images-contours-and-fields-plot-streamplot-py
# inspired by
# https://geoana.simpeg.xyz/auto_examples/em/plot_fdem_ElectricDipoleWholeSpace.html#sphx-glr-auto-examples-em-plot-fdem-electricdipolewholespace-py


def E_focus_density_plot(
    component,
    plane,
    **kwargs,
):

    optics = kwargs.get('optics', FocussingOptics())
    num_points = kwargs.get('num_points', 60)
    lim = kwargs.get('lim', 2)
    unit = kwargs.get('unit', Volt**2 / μm**2)
    scale = kwargs.get('scale', 'linear')
    num_contours = kwargs.get('num_contours', 60)
    # other choices: viridis_r, jet, jet_r
    cmap_choice = kwargs.get(
        'cmap_choice',
        {
            'linear': colourmap.viridis,
            'log': colourmap.viridis,
        }[scale],
    )
    title_str = kwargs.get(
        'title_str',
        {
            'x': r'$E_{\mathregular{focus}, x}^2$',
            'y': r'$E_{\mathregular{focus}, y}^2$',
            'z': r'$E_{\mathregular{focus}, z}^2$',
            'magnitude': r'$|\mathbf{E}_{\mathregular{focus}}|^2$',
        }[component],
    )
    use_pcolormesh = kwargs.get('use_pcolormesh', False)


    λ = optics.λ
    HORIZONTAL_AXIS_VALS, VERTICAL_AXIS_VALS = np.meshgrid(
        np.linspace(-lim, lim, num_points) * λ,
        np.linspace(-lim, lim, num_points) * λ,
    )
    HH, VV = HORIZONTAL_AXIS_VALS, VERTICAL_AXIS_VALS

    fig, ax = plt.subplots(figsize=(3, 3), dpi=250)
    ax.set_title(title_str)

    if plane == 'xy':
        h_label, v_label = r'$x$', r'$y$'
        spatio_temporal_args = (HH, VV, 0, 0)
    if plane == 'xz':
        h_label, v_label = r'$x$', r'$z$'
        spatio_temporal_args = (HH, 0, VV, 0)
    if plane == 'yz':
        h_label, v_label = r'$y$', r'$z$'
        spatio_temporal_args = (0, HH, VV, 0)

    density_values = optics.E_focus(component, *spatio_temporal_args)**2 / unit

    if component == 'magnitude' and kwargs.get('streamlines', False):
        h_component = np.abs(optics.E_focus(plane[0], *spatio_temporal_args))**2 / unit
        v_component = np.abs(optics.E_focus(plane[1], *spatio_temporal_args))**2 / unit

        #  Varying line width along a streamline
        size = np.sqrt(h_component**2 + v_component**2)
        lw = 1.5*size / size.max()
        ax.streamplot(
            HH  / λ, VV  / λ,
            h_component.T, v_component.T,
            # broken_streamlines=False,
            linewidth=lw,
            color=white,
            # color=size,
            # cmap='jet',
            arrowsize=0.5,
            arrowstyle='fancy',
            density=3.3,
        )

    if scale == 'linear':
        if use_pcolormesh:
            ax.pcolormesh(
                HORIZONTAL_AXIS_VALS / λ,
                VERTICAL_AXIS_VALS / λ,
                density_values,
                cmap=cmap_choice,
            )
        else:
            _ = ax.contourf(
                HORIZONTAL_AXIS_VALS / λ,
                VERTICAL_AXIS_VALS / λ,
                density_values,
                num_contours,
                cmap=cmap_choice,
            )
            # cbar = fig.colorbar(_, shrink=0.7, label=r'$(\mathregular{V/\mu m})^2$')
            # # cbar.locator = ticker.LogLocator(10)
            # cbar.set_ticks(cbar.locator.tick_values(density_values.min(), density_values.max()))
            # cbar.minorticks_off()

    elif scale == 'log':
        _ = ax.contourf(
            HORIZONTAL_AXIS_VALS / λ,
            VERTICAL_AXIS_VALS / λ,
            density_values,
            log_levels(density_values, num_contours),
            locator=ticker.LogLocator(),
            cmap=cmap_choice,
        )
        # cbar = fig.colorbar(_, shrink=0.7, label=r'$(\mathregular{V}/\mu m)^2$')
        # cbar.locator = ticker.LogLocator(10)
        # cbar.set_ticks(cbar.locator.tick_values(density_values.min(), density_values.max()))
        # cbar.minorticks_off()


    ax.set(aspect=1, xticklabels=[], yticklabels=[])
    ax.tick_params(right=False, left=False)
    ax.text(-1, -1.7, h_label, color=white)
    ax.annotate(
        '',
        xytext=(-1, -1.65),
        xy=(-1.75, -1.65),
        arrowprops=dict(arrowstyle='<-', color=white),
    )
    ax.text(-1.7, -1, v_label, color=white)
    ax.annotate('',
        xytext=(-1.65, -1),
        xy=(-1.65, -1.75),
        arrowprops=dict(arrowstyle='<-', color=white),
    )
    plt.show()

    return density_values

all_density_values = {}
if 1:
    for scale in (
        'linear',
        'log',
    ):
        for plane in (
            'xy',
            'xz',
            'yz',
        ):
            for component in (
                'x',
                'y',
                'z',
                'magnitude',
            ):
                d_vals = E_focus_density_plot(component, plane, scale=scale)
                all_density_values[component + '_' + plane] = d_vals

for key, d_vals in all_density_values.items():
    print(key, np.min(d_vals), np.max(d_vals))

In [ ]:
# x-axis, y=0
x, y = 1, 0
φ = np.arctan2(y, x)
print(φ/π, np.cos(2*φ), np.sin(2*φ), np.cos(φ))

# y-axis, x=0
x, y = 0, 1
φ = np.arctan2(y, x)
print(φ/π, np.cos(2*φ), np.sin(2*φ), np.cos(φ))

# 45 degrees in x-y plane
x, y = 1, 1
φ = np.arctan2(y, x)
print(φ/π, np.cos(2*φ), np.sin(2*φ), np.cos(φ))

# z-axis, x=y=0
x, y = 0, 0
φ = np.arctan2(y, x)
print(φ/π, np.cos(2*φ), np.sin(2*φ), np.cos(φ))

# Trapping potential and forces

In [ ]:
trap = Trap()

mosaic = [
    ['U'],
    ['F'],
]
fig = plt.figure(figsize=(FS, 3.5*len(mosaic)), dpi=250)
ax_dict = fig.subplot_mosaic(mosaic, gridspec_kw={'hspace': 0.5})

lim = 4
num_points = 200
X = np.linspace(-lim, lim, num_points) * μm
Y = np.linspace(-lim, lim, num_points) * μm
Z = np.linspace(-lim, lim, num_points) * μm

ax = ax_dict['U']
ax.set_title('Tweezer trapping potential vs. $x,y,z$')
U_unit = eV
ax.set_xlim(-lim, lim)
ax.set_ylim(-5, 0.01)
ax.set_xlabel(r'$x,y,z$ [$\mathregular{\mu m}$]')
ax.set_ylabel(r'[eV]')
ax.plot(X / μm, [trap.Utw(x, 0, 0) / U_unit for x in X], lw=2.00, alpha=0.8)
ax.plot(Y / μm, [trap.Utw(0, y, 0) / U_unit for y in Y], lw=1.50, alpha=0.8)
ax.plot(Z / μm, [trap.Utw(0, 0, z) / U_unit for z in Z], lw=1.25, alpha=0.8)
ax.vlines(trap.z_eq / μm, -5.5, 0.1, color=green, ls='--', lw=0.8, alpha=0.5)
ax.vlines([+trap.tweezer.zR / μm, -trap.tweezer.zR / μm], -5.5, 0.1, color=grey, ls='--', lw=0.8, alpha=0.5)


ax = ax_dict['F']
ax.set_title('Tweezer trapping forces vs. $x,y,z$')
F_unit = fN
ax.set_xlim(-lim, lim)
ax.set_ylim(-500, 500)
ax.set_xlabel(r'$x,y,z$ [$\mathregular{\mu m}$]')
ax.set_ylabel(r'[fN]')
ax.plot(X / μm, [trap.Ftwx(x, 0, 0) / F_unit for x in X], lw=2.00, alpha=0.8)
ax.plot(Y / μm, [trap.Ftwy(0, y, 0) / F_unit for y in Y], lw=1.50, alpha=0.8)
ax.plot(Z / μm, [trap.Ftwz(0, 0, z) / F_unit for z in Z], lw=1.25, alpha=0.8)
ax.vlines(trap.z_eq / μm, -510, 510, color=green, ls='--', lw=0.8, alpha=0.5)
ax.vlines([+trap.tweezer.zR / μm, -trap.tweezer.zR / μm], -510, 510, color=grey, ls='--', lw=0.8, alpha=0.5)

plt.show()


# Frequencies of tweezer potential

In [ ]:
test_bead = Bead(
    εR=2.05,
    # εR=1.98,
    # εR=2.10,
    r=70*nm,
    ρ=1850.0,
)
test_bead.info

trap_test = Trap(
    tweezer=Tweezer(
        # Power=485*mW,
        # wy0=1.051*μm,
        # wx0=0.938*μm,

        Power=500*mW,
        wy0=1.190*μm,
        wx0=1.004*μm,

        # wy0=1.18*μm,
        # wx0=1.09*μm,

        # Power=200*mW,
        # wy0=1.285*μm,
        # wx0=1.116*μm,

        # Power=1000*mW,
        # wy0=1.356*μm,
        # wx0=1.106*μm,
    ),
    bead=test_bead,
)

for trap in (Trap(), trap_test):
    print()
    print(f'Ωx = {trap.Ωx / kHz:.1f} kHz')
    print(f'Ωy = {trap.Ωy / kHz:.1f} kHz')
    print(f'Ωz = {trap.Ωz / kHz:.1f} kHz')
    print(f'z_eq = {trap.z_eq / μm:.3f} μm')

# Animate contour plots

In [ ]:
# NOTE: how to do the log scale and colourbar
# https://stackoverflow.com/questions/65823932/plt-contourf-with-given-number-of-levels-in-logscale

# NOTE: how to do the streamline plots
# https://matplotlib.org/stable/gallery/images_contours_and_fields/plot_streamplot.html#sphx-glr-gallery-images-contours-and-fields-plot-streamplot-py
# inspired by
# https://geoana.simpeg.xyz/auto_examples/em/plot_fdem_ElectricDipoleWholeSpace.html#sphx-glr-auto-examples-em-plot-fdem-electricdipolewholespace-py


fig, ax = plt.subplots(figsize=(3, 3), dpi=300)

optics = FocussingOptics()
T = optics.T
step = 0.2 * T
times = np.arange(0, T + step, step)

frames = []
component, plane = 'magnitude', 'xy'

step = 500
fs = np.arange(500.25, 4000.25 + step, step) * optics.λ

step = 0.05
NAs = np.arange(0.2, 1.0 + step, step)

t = 0
ii = 0

# for t in times:
# for f in fs:
for NA in preview_frames(NAs):
    ii += 1
    frame = []
    kwargs = dict(
        # fo_kwargs=dict(focal_length=f),
        fo_kwargs=dict(NA=NA),
        scale='log',
        streamlines=True,
    )

    fo_kwargs = kwargs.get('fo_kwargs', {})
    optics = kwargs.get('optics', FocussingOptics(**fo_kwargs))
    num_points = kwargs.get('num_points', 50)
    lim = kwargs.get('lim', 2)
    unit = kwargs.get('unit', Volt**2 / μm**2)
    scale = kwargs.get('scale', 'linear')
    num_contours = kwargs.get('num_contours', 50)
    # other choices: viridis_r, jet, jet_r
    cmap_choice = kwargs.get(
        'cmap_choice',
        {
            'linear': colourmap.viridis,
            'log': colourmap.viridis,
        }[scale],
    )
    title_str = kwargs.get(
        'title_str',
        {
            'x': r'$E_{\mathregular{focus}, x}^2$',
            'y': r'$E_{\mathregular{focus}, y}^2$',
            'z': r'$E_{\mathregular{focus}, z}^2$',
            'magnitude': r'$|\mathbf{E}_{\mathregular{focus}}|^2$',
        }[component],
    )
    title_str = r'$\mathrm{BOOGIE}$ @ $\mathrm{BAUGÉ}$'
    use_pcolormesh = kwargs.get('use_pcolormesh', False)

    λ = optics.λ
    HORIZONTAL_AXIS_VALS, VERTICAL_AXIS_VALS = np.meshgrid(
        np.linspace(-lim, lim, num_points) * λ,
        np.linspace(-lim, lim, num_points) * λ,
    )
    HH, VV = HORIZONTAL_AXIS_VALS, VERTICAL_AXIS_VALS

    ax.set_title(title_str)
    if plane == 'xy':
        h_label, v_label = r'$x$', r'$y$'
        spatio_temporal_args = (HH, VV, 0, t)
    if plane == 'xz':
        h_label, v_label = r'$x$', r'$z$'
        spatio_temporal_args = (HH, 0, VV, t)
    if plane == 'yz':
        h_label, v_label = r'$y$', r'$z$'
        spatio_temporal_args = (0, HH, VV, t)

    h_label, v_label = r'$j$', r'$e$'

    density_values = optics.E_focus(component, *spatio_temporal_args)**2 / unit

    if component == 'magnitude' and kwargs.get('streamlines', False):
        h_component = np.abs(optics.E_focus(plane[0], *spatio_temporal_args))**2 / unit
        v_component = np.abs(optics.E_focus(plane[1], *spatio_temporal_args))**2 / unit

        #  Varying line width along a streamline
        size = np.sqrt(h_component**2 + v_component**2)
        lw = 1.5*size / size.max()
        sp = ax.streamplot(
            HH  / λ, VV  / λ,
            h_component.T, v_component.T,
            # broken_streamlines=False,
            linewidth=lw,
            color=white,
            # color=size,
            # cmap='jet',
            arrowsize=0.5,
            arrowstyle='fancy',
            density=2.5,
        )
        # frame.append(sp.lines)
        # frame.append(sp.arrows)

    if scale == 'linear':
        if use_pcolormesh:
            ax.pcolormesh(
                HORIZONTAL_AXIS_VALS / λ,
                VERTICAL_AXIS_VALS / λ,
                density_values,
                cmap=cmap_choice,
            )
        else:
            cf = ax.contourf(
                HORIZONTAL_AXIS_VALS / λ,
                VERTICAL_AXIS_VALS / λ,
                density_values,
                num_contours,
                cmap=cmap_choice,
            )
            # https://stackoverflow.com/questions/61534888/artistanimation-using-contourf-for-model-plots
            # Matplotlib >= 3.8: the ContourSet is itself the artist (ContourSet.collections was removed in 3.10)
            frame.append(cf)
            # frame.append(cf.collections)

    elif scale == 'log':
        cf = ax.contourf(
            HORIZONTAL_AXIS_VALS / λ,
            VERTICAL_AXIS_VALS / λ,
            density_values,
            log_levels(density_values, num_contours),
            locator=ticker.LogLocator(),
            cmap=cmap_choice,
        )
        # Matplotlib >= 3.8: the ContourSet is itself the artist (ContourSet.collections was removed in 3.10)
        frame.append(cf)
        # frame.append(cf.collections)

    pos = np.linspace(-5, 2, 100)
    _ = ax.text(np.random.choice(pos) / 3, np.random.choice(pos) / 3, '¡RSVP!'[:ii // 2], fontsize=20, color=red)
    frame.append(_)
    _ = ax.text(np.random.choice(pos) / 3, np.random.choice(pos) / 3, '¡PSVR!'[:ii // 2][::-1], fontsize=15, color=white)
    frame.append(_)
    _ = ax.text(np.random.choice(pos) / 3, np.random.choice(pos) / 3, '!¡PSVR¡¡'[:ii // 2][::-1], fontsize=12, color=RGB_magenta)
    frame.append(_)

    frames.append(frame)

col = np.random.choice(list(all_colours_dict.values()))
ax.set(aspect=1, xticklabels=[], yticklabels=[])
ax.tick_params(right=False, left=False)
ax.text(-1, -1.7, h_label, color=col, fontsize=15)
ax.annotate(
    '',
    xytext=(-1, -1.65),
    xy=(-1.75, -1.65),
    arrowprops=dict(arrowstyle='<-', color=col),
)
ax.text(-1.7, -1, v_label, color=col, fontsize=15)
ax.annotate('',
    xytext=(-1.65, -1),
    xy=(-1.65, -1.75),
    arrowprops=dict(arrowstyle='<-', color=col),
)

ani = ArtistAnimation(fig, frames, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, 'Animate contour plots')
